# Fase 1. Comprensión y Preparación de los Datos

## Contexto del negocio

La empresa objeto de este análisis es una distribuidora de suplementos vitamínicos que opera bajo un modelo de distribución mayorista y minorista a nivel nacional. Sus canales de venta incluyen farmacias, droguerías independientes y grandes cadenas comerciales, lo que le permite alcanzar un mercado amplio y heterogéneo en términos de volumen de compra y frecuencia de pedido.

Dada la naturaleza perecedera de sus productos y la extensión de su portafolio, la gestión eficiente del inventario representa un reto crítico para la organización. El presente notebook desarrolla la primera fase de la metodología CRISP-DM: la extracción, exploración y preparación de los datos transaccionales (pedidos, ventas y devoluciones) provenientes del sistema ERP SAP HANA, como base para las fases posteriores de clasificación de inventario y predicción de demanda.

> **Nota de confidencialidad.** Esta versión del notebook se publica sin datos ni salidas. Los nombres de servidor, base de datos, archivos internos, bodegas y patrones de codificación de la empresa se reemplazaron por parámetros genéricos (celda *Parámetros específicos del entorno*). Las celdas de interpretación explican cómo leer cada resultado y qué decisión se toma con él, sin cifras del negocio.

# A1.1 — Extracción y análisis exploratorio (EDA)

### Extracción de datos desde SAP HANA

Para la construcción de la Tabla Base Analítica se diseñó un flujo de extracción que conecta cinco fuentes del ERP SAP Business One, alojado sobre SAP HANA, mediante consultas SQL nativas. Las fuentes extraídas son: Ventas, Pedidos, Devoluciones, el maestro de Productos y una tabla de cierre diario de inventario por producto. Cada una de estas consultas se materializa como una tabla independiente dentro del modelo, que posteriormente se integra y transforma en Power BI Service para dar forma a la TBA.

El flujo general de extracción sigue el siguiente recorrido:

1. Conexión a SAP HANA: se establece la conexión al servidor, donde reside la base de datos correspondiente al ERP de la distribuidora.
2. Consulta SQL por fuente: para cada una de las cinco entidades de negocio (pedidos, ventas, devoluciones, inventario y productos) se define una consulta SQL que extrae únicamente los campos necesarios para el análisis, evitando traer columnas irrelevantes que incrementen el volumen de datos sin aportar valor analítico.
3. Carga a Power Query: cada consulta se conecta mediante el conector nativo SapHana.Database de Power Query, que traduce el resultado de la consulta SQL en una tabla consumible dentro del modelo de datos.
4. Publicación en Power BI Service: una vez validado el modelo en Power BI Desktop, se publica en el servicio, donde se configura la actualización programada contra el gateway de datos local, garantizando que la ABT se mantenga sincronizada con la operación diaria del ERP.

A continuación, se detalla la consulta correspondiente a cada fuente.

> **Nota sobre privacidad de los datos.** Por confidencialidad, los nombres del servidor, de la base de datos y del campo de usuario de la línea comercial se reemplazaron por identificadores genéricos (`SERVIDOR_SAP_HANA`, `SBO_EMPRESA`, `U_LineaComercial`). La estructura y la lógica de las consultas son las del entorno productivo; solo cambia la información propia de la infraestructura de la empresa.

**Pedidos**

```m
SapHana.Database("SERVIDOR_SAP_HANA", [Implementation = "2.0", Query = "
    SELECT
        T0.""DocEntry"",
        T0.""DocNum"",
        T0.""DocDate"",
        T0.""DocDueDate"",
        T0.""DocType"",
        T0.""CardCode"",
        T0.""CardName"",
        T0.""DocStatus"",
        T0.""CANCELED"",
        T1.""LineNum"",
        T1.""ItemCode"",
        T1.""Quantity"",
        T1.""DelivrdQty"",
        T1.""OpenQty"",
        T1.""unitMsr"",
        T1.""NumPerMsr"",
        T1.""PriceBefDi"",
        T1.""DiscPrcnt"",
        T1.""Price"",
        T1.""LineTotal"",
        T1.""WhsCode"",
        T1.""LineStatus""
    FROM SBO_EMPRESA.ORDR T0
    JOIN SBO_EMPRESA.RDR1 T1 ON T1.""DocEntry"" = T0.""DocEntry""
    "])
```

Esta consulta extrae los pedidos de venta desde las tablas estándar de SAP Business One ORDR (encabezado del documento) y RDR1 (líneas del documento), unidas por DocEntry. El encabezado aporta la información general del pedido (fecha, cliente, estado, si fue cancelado), mientras que las líneas aportan el detalle por producto: ítem, cantidad, precio, valor total de línea, bodega (WhsCode) y estado de la línea (LineStatus), este último relevante para identificar líneas cerradas o pendientes de entrega. Se incluyen además la cantidad entregada y pendiente (DelivrdQty, OpenQty), la unidad de medida (unitMsr, NumPerMsr) y el precio antes de descuento con su porcentaje (PriceBefDi, DiscPrcnt), que se usan para medir el nivel de servicio y la actividad promocional.

**Ventas**

```m
SapHana.Database("SERVIDOR_SAP_HANA", [Implementation = "2.0", Query = "
    SELECT
        T0.""DocEntry"",
        T0.""DocNum"",
        T0.""DocDate"",
        T0.""DocType"",
        T0.""CardCode"",
        T0.""CardName"",
        T0.""DocStatus"",
        T0.""CANCELED"",
        T1.""LineNum"",
        T1.""ItemCode"",
        T1.""Quantity"",
        T1.""Price"",
        T1.""LineTotal"",
        T1.""WhsCode"",
        T1.""BaseType"",
        T1.""BaseEntry"",
        T1.""BaseLine""
    FROM SBO_EMPRESA.OINV T0
    JOIN SBO_EMPRESA.INV1 T1 ON T1.""DocEntry"" = T0.""DocEntry""
    "])
```
Esta consulta extrae las facturas de venta desde OINV (encabezado) e INV1 (líneas), también unidas por DocEntry. A diferencia de la consulta de Pedidos, aquí se incluyen los campos BaseType, BaseEntry y BaseLine, que permiten trazar cada línea de factura hacia el documento origen del cual proviene (por ejemplo, el pedido que le dio lugar), habilitando en fases posteriores el cruce entre lo pedido y lo efectivamente facturado.

**Devoluciones**

```m
SapHana.Database("SERVIDOR_SAP_HANA", [Implementation = "2.0", Query = "
    SELECT
        T0.""DocEntry"",
        T0.""DocNum"",
        T0.""DocDate"",
        T0.""DocType"",
        T0.""CardCode"",
        T0.""CardName"",
        T0.""DocStatus"",
        T0.""CANCELED"",
        T1.""LineNum"",
        T1.""ItemCode"",
        T1.""Quantity"",
        T1.""Price"",
        T1.""LineTotal"",
        T1.""WhsCode"",
        T1.""BaseType"",
        T1.""BaseEntry"",
        T1.""BaseLine""
    FROM SBO_EMPRESA.ORIN T0
    JOIN SBO_EMPRESA.RIN1 T1 ON T1.""DocEntry"" = T0.""DocEntry""
    "])

```
Esta consulta extrae las devoluciones de mercancía desde ORIN (encabezado) y RIN1 (líneas), con una estructura simétrica a la de Ventas. Al igual que en esa consulta, se incluyen BaseType, BaseEntry y BaseLine para trazar cada devolución hasta la factura que le dio origen, lo que permite calcular en fases posteriores la venta neta real por producto (ventas menos devoluciones asociadas).

**Inventario (cierre diario de existencias)**

```m
SapHana.Database("SERVIDOR_SAP_HANA", [Implementation = "2.0", Query = "
        WITH movements AS (
            SELECT 
                T0.""ItemCode"",
                T0.""Warehouse"",
                T0.""DocDate"",
                T0.""CreateDate"",
                T0.""TransNum"",
                (T0.""InQty"" - T0.""OutQty"") AS ""Quantity"",
                SUM(T0.""InQty"" - T0.""OutQty"") OVER (
                    PARTITION BY T0.""ItemCode"", T0.""Warehouse""
                    ORDER BY T0.""DocDate"", T0.""CreateDate"", T0.""TransNum""
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ) AS ""RunningBalance""
            FROM SBO_EMPRESA.OINM T0
        ),
        daily_close AS (
            SELECT 
                ""ItemCode"",
                ""Warehouse"",
                ""DocDate"",
                ""RunningBalance"",
                ROW_NUMBER() OVER (
                    PARTITION BY ""ItemCode"", ""Warehouse"", ""DocDate""
                    ORDER BY ""CreateDate"" DESC, ""TransNum"" DESC
                ) AS rn
            FROM movements
        )
        SELECT 
            dc.""ItemCode"",
            T1.""ItemName"",
            dc.""Warehouse"",
            T2.""WhsName"",
            dc.""DocDate""        AS ""FechaCierre"",
            dc.""RunningBalance"" AS ""StockCierreDia""
        FROM daily_close dc
        JOIN SBO_EMPRESA.OITM T1 ON T1.""ItemCode"" = dc.""ItemCode""
        JOIN SBO_EMPRESA.OWHS T2 ON T2.""WhsCode"" = dc.""Warehouse""
        WHERE dc.rn = 1
        ORDER BY dc.""ItemCode"", dc.""Warehouse"", dc.""DocDate""
        "])
```

Esta es la consulta más elaborada del flujo, ya que SAP HANA no almacena directamente el saldo de inventario por día: solo registra los movimientos individuales en OINM (entradas y salidas de mercancía). Para reconstruir el stock de cierre diario por producto y bodega, la consulta se resuelve en dos pasos mediante CTEs (Common Table Expressions):

- movements: 
calcula, para cada movimiento de inventario, el saldo acumulado (RunningBalance) hasta ese momento, mediante una función de ventana (SUM(...) OVER (PARTITION BY ItemCode, Warehouse ORDER BY DocDate, CreateDate, TransNum)). Esto genera una especie de "saldo corriente" que crece o decrece con cada entrada o salida, particionado por producto y bodega.
- daily_close: 
de todos los movimientos ocurridos en un mismo día para un mismo producto y bodega, selecciona únicamente el último (ROW_NUMBER() ... ORDER BY CreateDate DESC, TransNum DESC y filtro rn = 1), que representa el saldo con el que ese producto cerró el día.

El resultado final une este cierre diario con el maestro de productos (OITM) y de bodegas (OWHS) para incorporar los nombres descriptivos (ItemName, WhsName), entregando una tabla con el stock de cierre por producto, bodega y fecha (StockCierreDia). Esta tabla será clave más adelante para identificar quiebres de stock y contextualizar los patrones de demanda.

**Productos**

```m
SapHana.Database("SERVIDOR_SAP_HANA", [Implementation = "2.0", Query = "
    SELECT
        T0.""ItemCode"",
        T0.""ItemName"",
        T0.""ItmsGrpCod"",
        T1.""ItmsGrpNam"",
        T0.""SellItem"",
        T0.""InvntItem"",
        T0.""validFor"",
        T0.""AvgPrice"",
        T0.""OnHand"",
        T0.""CreateDate"",
        T0.""UpdateDate"",
        T0.""U_LineaComercial"" AS ""NombreLinea""
    FROM SBO_EMPRESA.OITM T0
    LEFT JOIN SBO_EMPRESA.OITB T1 ON T1.""ItmsGrpCod"" = T0.""ItmsGrpCod""
    WHERE T0.""SellItem"" = 'Y' and
    T1.""ItmsGrpNam"" like 'PT%'
    "])
```

Esta consulta extrae el maestro de artículos desde OITM, enriquecido con el nombre del grupo de artículos (ItmsGrpNam) mediante un LEFT JOIN contra OITB. Se aplican dos filtros relevantes para acotar el alcance del proyecto: SellItem = 'Y', que descarta artículos no vendibles (por ejemplo, insumos internos), y ItmsGrpNam LIKE 'PT%', que restringe el resultado a los grupos de artículos cuyo nombre inicia con "PT" (producto terminado), excluyendo materias primas u otras categorías que no son objeto de la distribución comercial. Adicionalmente, se incluyen campos como AvgPrice y OnHand, que aportan una fotografía puntual del costo promedio y la existencia actual del artículo al momento de la extracción, y el campo de usuario de la línea comercial (`NombreLinea`).

**Tabla de equivalencias de códigos.** Además de las consultas al ERP, se incorporó un registro de equivalencias de códigos, mantenido por el área de producto en un archivo de Excel almacenado en el Lakehouse (`RUTA_EQUIV`). Este archivo documenta los productos cuyo código fue reemplazado durante el periodo analizado y se utiliza para homologar las series históricas (sección 1.1.0).

### Del flujo de extracción al análisis.
Las consultas anteriores, ejecutadas a través de Power Query, no alimentan directamente el entorno de análisis. Su función es extraer la información desde SAP HANA y almacenarla en un Data Lake de Azure, que actúa como capa de persistencia intermedia entre el ERP y los procesos analíticos. Este desacople tiene una razón práctica, ya que evita ejecutar consultas repetidas contra el ERP productivo para cada análisis y centraliza los datos crudos en un único repositorio versionado, disponible para distintos consumidores como reportes de Power BI, notebooks de análisis y futuros modelos.

A partir de este punto, el presente notebook se conecta directamente al Data Lake, no a SAP HANA, para leer los datos ya extraídos de Pedidos, Ventas, Devoluciones, Productos e Inventario, y realizar sobre ellos el análisis exploratorio utilizando PySpark. Esto permite aprovechar el procesamiento distribuido de Spark para trabajar con los volúmenes de datos transaccionales de la distribuidora de forma eficiente.

### 1.1 Carga de datos desde el Data Lake

Con los datos ya extraídos y almacenados en el Data Lake, el siguiente paso es cargarlos dentro del notebook de Fabric para iniciar el análisis exploratorio con PySpark. El lakehouse contiene cinco tablas, correspondientes a las fuentes descritas en la sección anterior. D_Productos almacena el maestro de artículos, F_Pedidos y F_Ventas contienen los pedidos y las facturas de venta respectivamente, F_Devoluciones recoge las devoluciones asociadas a esas ventas, y F_Inventario contiene el stock de cierre diario por producto y bodega.

El siguiente bloque configura el entorno de trabajo (librerías de manipulación y visualización de datos) y carga cada una de estas tablas como un DataFrame de Spark independiente, a partir de un diccionario de referencia que centraliza los nombres de las tablas de origen.

In [ ]:
# ============================================================================
# FASE 1: COMPRENSIÓN Y PREPARACIÓN DE LOS DATOS
# Autor: Daniel Escobar Cardona
#
# Tablas de origen (lakehouse):
#   Productos    -> Maestro de artículos (OITM)
#   Pedidos      -> Pedidos de venta (ORDR/RDR1)
#   Ventas       -> Facturas (OINV/INV1)
#   Devoluciones -> Devoluciones (ORIN/RIN1)
#   Inventario   -> Stock de cierre diario (OINM)
# ============================================================================

# Librerias
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Graficas
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

# Diccionario de tablas
TABLES = {
    "pedidos": "Pedidos",
    "ventas": "Ventas",
    "devoluciones": "Devoluciones",
    "inventario": "Inventario",
    "productos": "Productos",
}

# Asignación de variables  
df_pedidos = spark.table(TABLES["pedidos"]).cache()
df_ventas = spark.table(TABLES["ventas"]).cache()
df_devoluciones = spark.table(TABLES["devoluciones"]).cache()
df_inventario = spark.table(TABLES["inventario"]).cache()
df_productos = spark.table(TABLES["productos"]).cache()

#### Parámetros específicos del entorno

Los valores que dependen de la infraestructura o de la codificación interna de la empresa (rutas, códigos de bodega, patrones de codificación y líneas excluidas) se concentran en la celda siguiente. En este repositorio tienen valores de ejemplo; en el entorno privado se reemplazan por los reales. El resto de los parámetros (umbrales y ventanas) son decisiones metodológicas y se definen en la sección donde se usan.

In [ ]:
# ----------------------------------------------------------------------------
# Parámetros específicos del entorno y del negocio (valores de ejemplo)
# ----------------------------------------------------------------------------
# Archivo de equivalencias de códigos (código anterior -> código vigente) en el Lakehouse
RUTA_EQUIV       = "/lakehouse/default/Files/equivalencias_codigos.xlsx"
HOJA_EQUIV       = "equivalencias"
COLS_EQUIV       = [2, 5, 6]              # posiciones: código anterior (C), código nuevo (F), descripción (G)
ENCABEZADO_EQUIV = "Número de artículo"   # encabezado repetido dentro de la hoja, se descarta

# Identificación de muestras gratuitas dentro del maestro de producto terminado
PATRON_NOMBRE_MUESTRA = r"GRATIS|MUESTRA"  # palabras en la descripción del artículo
PATRON_CODIGO_MUESTRA = r"-M$"             # sufijo de código reservado para muestras (ejemplo)

# Bodegas (se definen con el área de operaciones a partir del análisis de la sección 1.7)
BODEGAS_VENTA   = ["01", "02", "03"]       # inventario disponible para la venta (ejemplo)
BODEGA_VENCIDOS = "99"                     # producto vencido (ejemplo)

# Líneas comerciales excluidas del alcance: producción por pedido (sección 1.8)
LINEAS_FUERA = ["Maquilas"]

# Revisión interna: muestra nombres de clientes en las tablas de control.
# Debe quedar en False en cualquier versión que se exporte o publique.
REVISION_INTERNA = False

Con las cinco tablas cargadas y cacheadas en memoria, se da por completada la carga inicial de datos desde el Data Lake, quedando el insumo listo para el análisis exploratorio. Antes de entrar al detalle de cada fuente, se realiza una verificación general del esquema y, a continuación, un primer perfilamiento conjunto de rangos de fecha y volumen de registros por fuente.

In [ ]:
# ----------------------------------------------------------------------------
# 0. Verificación rápida de esquema, tipos y versión de los datos
# ----------------------------------------------------------------------------
FUENTES = {
    "Pedidos": df_pedidos,
    "Ventas": df_ventas,
    "Devoluciones": df_devoluciones,
    "Inventario": df_inventario,
    "Productos": df_productos,
}

# 0.a Esquema consolidado: (filas) x (columnas)
filas = [
    {"fuente": fuente, "columna": c.name, "tipo": c.dataType.simpleString()}
    for fuente, df in FUENTES.items()
    for c in df.schema.fields
]
pd_esquema = (
    pd.DataFrame(filas)
      .pivot_table(index="columna", columns="fuente", values="tipo", aggfunc="first")
      .reindex(columns=list(FUENTES))
      .fillna("—")
)
display(pd_esquema.reset_index())

# 0.b Estructura compartida entre las tablas documento (encabezado-línea)
cols_ped, cols_ven, cols_dev = (set(df.columns) for df in (df_pedidos, df_ventas, df_devoluciones))
print("Columnas comunes a las 3 tablas documento:", sorted(cols_ped & cols_ven & cols_dev))
print("Solo en Pedidos:                          ", sorted(cols_ped - cols_ven))
print("Solo en Ventas/Devoluciones:              ", sorted((cols_ven & cols_dev) - cols_ped))

# 0.c Volumen y versión Delta de cada tabla (para fijar el snapshot del análisis)
resumen_version = []
for fuente, df in FUENTES.items():
    tabla = TABLES[fuente.lower()]
    try:
        h = spark.sql(f"DESCRIBE HISTORY {tabla}").orderBy(F.desc("version")).first()
        version, ts = h["version"], h["timestamp"]
    except Exception:
        version, ts = None, None
    resumen_version.append({
        "fuente": fuente, "tabla": tabla, "filas": df.count(),
        "columnas": len(df.columns), "version_delta": version, "ultima_escritura": ts,
    })
pd_version = pd.DataFrame(resumen_version)
display(pd_version)

El esquema confirma que las tres tablas documento (Pedidos, Ventas y Devoluciones) comparten la estructura encabezado‑línea propia de SAP Business One: identificador del documento (DocEntry, LineNum), fecha (DocDate), cliente (CardCode), artículo (ItemCode), cantidad, precio, total de línea y bodega. Ventas y Devoluciones incorporan además los campos de trazabilidad BaseType, BaseEntry y BaseLine, que vinculan cada línea con su documento de origen. Pedidos, por su parte, incluye DocDueDate y LineStatus, propios del ciclo de vida del pedido.

Los campos de cantidad y valor llegan como decimal (34,6), el tipo de alta precisión de SAP HANA. Se conservarán así en los cálculos monetarios y se convertirán a double solo en las agregaciones y visualizaciones. Las fechas llegan como timestamp sin componente horario relevante, por lo que se truncarán a día o mes según la granularidad de cada análisis.

La extracción incorpora además el campo DocType del encabezado, que en SAP Business One distingue los documentos de artículo (I) de los documentos de servicio (S). En Pedidos se incluyen también la cantidad entregada (DelivrdQty), la cantidad pendiente (OpenQty), la unidad de medida (unitMsr, NumPerMsr), el porcentaje de descuento (DiscPrcnt) y el precio antes de descuento (PriceBefDi), que permiten medir el nivel de servicio y la actividad promocional en las secciones posteriores.

In [ ]:
# ----------------------------------------------------------------------------
# 1.1 Cobertura temporal y ventana de análisis
# ----------------------------------------------------------------------------

# 1.1.a Fijar snapshot de datos (versiones Delta tomadas de la celda 0).
#       Se escriben a mano a propósito: si se leyeran automáticamente,
#       cada ejecución tomaría la última versión y las cifras volverían a moverse.
VERSIONES = {
    "Pedidos": 6, "Ventas": 6, "Devoluciones": 6, 
    "Inventario": 6, "Productos": 7,
}

def leer_snapshot(tabla):
    return spark.sql(f"SELECT * FROM {tabla} VERSION AS OF {VERSIONES[tabla]}")

spark.catalog.clearCache()
df_pedidos      = leer_snapshot("Pedidos").cache()
df_ventas       = leer_snapshot("Ventas").cache()
df_devoluciones = leer_snapshot("Devoluciones").cache()
df_inventario   = leer_snapshot("Inventario").cache()
df_productos    = leer_snapshot("Productos").cache()

for nombre, df in [("Pedidos", df_pedidos), ("Ventas", df_ventas), ("Devoluciones", df_devoluciones)]:
    assert "DocType" in df.columns, f"{nombre} no tiene DocType: revisar la versión fijada"

#### Homologación de códigos de producto.

Durante el periodo analizado, la empresa reemplazó el código de un grupo de productos sin modificar el producto en sí (cambios de codificación asociados a una actualización de referencias). En el ERP, cada código conserva su propio historial, por lo que un mismo producto aparece fragmentado en dos series: la del código anterior, que se extingue, y la del código vigente, que inicia desde cero. Sin corrección, esto produce tres distorsiones: el portafolio aparece más extenso de lo que es, el código vigente se clasifica como producto nuevo con historia insuficiente, y la transición entre códigos se interpreta como volatilidad o desabastecimiento. Para evitarlo, a partir del registro de equivalencias que mantiene el área de producto se construyó una tabla de equivalencias, y todas las fuentes se homologaron al código vigente antes de cualquier análisis. La dirección de cada equivalencia se validó con los datos: se asume como vigente el código con el pedido más reciente.

In [ ]:
# ----------------------------------------------------------------------------
# 1.1.0 Homologación de códigos (cambio de referencia viejo -> nuevo)
# ----------------------------------------------------------------------------
EXCLUIR_PARES = []          # p. ej. [("COD_ANTERIOR", "COD_NUEVO")] si alguna equivalencia no aplica tras la revisión
DIAS_TOLERANCIA = 90        # solape aceptado entre el último pedido del viejo y el primero del nuevo

# 1.1.0.a Tabla de equivalencias desde el Excel (COLS_EQUIV: código anterior, código nuevo, descripción)
raw = pd.read_excel(RUTA_EQUIV, sheet_name=HOJA_EQUIV, header=None, dtype=str)
equiv = raw[COLS_EQUIV].dropna(subset=COLS_EQUIV[:2])
equiv.columns = ["cod_c", "cod_f", "descripcion"]
equiv[["cod_c", "cod_f"]] = equiv[["cod_c", "cod_f"]].apply(lambda s: s.str.strip())
equiv = (equiv[(equiv["cod_f"] != ENCABEZADO_EQUIV) & (equiv["cod_c"] != equiv["cod_f"])]
         .drop_duplicates(["cod_c", "cod_f"]).reset_index(drop=True))
equiv = equiv[~equiv[["cod_c", "cod_f"]].apply(tuple, axis=1).isin(EXCLUIR_PARES)]
print(f"Pares de equivalencia leídos: {len(equiv)}")

# 1.1.0.b Validación con los pedidos (vigentes, de artículo) del snapshot sin homologar
act = (df_pedidos.filter((F.col("DocType") == "I") & (F.col("CANCELED") == "N"))
       .groupBy("ItemCode")
       .agg(F.min("DocDate").alias("primero"), F.max("DocDate").alias("ultimo"),
            F.sum(F.col("Quantity").cast("double")).alias("unidades"))
       .toPandas())
v = (equiv.merge(act.add_suffix("_c"), left_on="cod_c", right_on="ItemCode_c", how="left")
          .merge(act.add_suffix("_f"), left_on="cod_f", right_on="ItemCode_f", how="left")
          .drop(columns=["ItemCode_c", "ItemCode_f"]))
tol = pd.Timedelta(days=DIAS_TOLERANCIA)

def relacion(r):
    c, f = pd.notna(r["ultimo_c"]), pd.notna(r["ultimo_f"])
    if not c and not f: return "Sin pedidos"
    if c and not f:     return "Solo código C"
    if f and not c:     return "Solo código F"
    if r["ultimo_c"] <= r["primero_f"] + tol: return "Secuencial C → F"
    if r["ultimo_f"] <= r["primero_c"] + tol: return "Secuencial F → C"
    return "Coexisten"

v["relacion"] = v.apply(relacion, axis=1)
# Destino = código con el pedido más reciente; sin pedidos en ninguno -> columna F
v["cod_destino"] = np.where(v["ultimo_c"].fillna(pd.Timestamp.min) > v["ultimo_f"].fillna(pd.Timestamp.min),
                            v["cod_c"], v["cod_f"])
v.loc[v["relacion"] == "Sin pedidos", "cod_destino"] = v["cod_f"]
v["cod_origen"] = np.where(v["cod_destino"] == v["cod_f"], v["cod_c"], v["cod_f"])

display(v["relacion"].value_counts().rename("pares").reset_index())
display(v[v["relacion"] != "Sin pedidos"]
        [["cod_c", "cod_f", "descripcion", "relacion", "cod_origen", "cod_destino",
          "primero_c", "ultimo_c", "unidades_c", "primero_f", "ultimo_f", "unidades_f"]]
        .sort_values("relacion"))

# 1.1.0.c Aplicación a todas las fuentes (se conserva el código original en ItemCode_orig)
mapa_sp = F.broadcast(spark.createDataFrame(v[["cod_origen", "cod_destino"]]))

def homologar(df):
    return (df.withColumnRenamed("ItemCode", "ItemCode_orig")
              .join(mapa_sp, F.col("ItemCode_orig") == F.col("cod_origen"), "left")
              .withColumn("ItemCode", F.coalesce("cod_destino", "ItemCode_orig"))
              .drop("cod_origen", "cod_destino"))

df_pedidos      = homologar(df_pedidos).cache()
df_ventas       = homologar(df_ventas).cache()
df_devoluciones = homologar(df_devoluciones).cache()
df_inventario   = homologar(df_inventario).cache()

# Maestro: una fila por código destino (atributos del código vigente; stock sumado; creación más antigua)
prod = homologar(df_productos)
w_p = Window.partitionBy("ItemCode").orderBy((F.col("ItemCode_orig") == F.col("ItemCode")).desc(),
                                              F.desc("UpdateDate"))
agg_p = prod.groupBy("ItemCode").agg(
    F.sum("OnHand").alias("OnHand"), F.min("CreateDate").alias("CreateDate"),
    F.concat_ws(",", F.array_sort(F.collect_set("ItemCode_orig"))).alias("codigos_origen"))
df_productos = (prod.withColumn("rn", F.row_number().over(w_p)).filter("rn = 1")
                    .drop("rn", "OnHand", "CreateDate", "ItemCode_orig")
                    .join(agg_p, "ItemCode").cache())

# 1.1.0.d Control
lineas_reasignadas = df_pedidos.filter(F.col("ItemCode") != F.col("ItemCode_orig")).count()
print(f"Líneas de pedido reasignadas a otro código: {lineas_reasignadas:,}")
print(f"SKUs en el maestro PT: {df_productos.count()} (antes de homologar: {prod.count()})")
pd_equivalencias = v.copy()     # se guarda en la sección de persistencia

**Cómo leer el resultado.** La primera tabla resume cuántos pares de equivalencia caen en cada tipo de relación: *secuencial* (el código anterior deja de pedirse cuando inicia el vigente), *coexisten* (ambos códigos registran pedidos durante más de `DIAS_TOLERANCIA` días, lo que suele corresponder al agotamiento del inventario rotulado con el código anterior), *solo un código* y *sin pedidos*. La segunda tabla permite revisar par por par el sentido de cada cambio; si alguna equivalencia no aplica, se retira con `EXCLUIR_PARES`.

El efecto principal de la homologación no es reducir el portafolio, sino recuperar historia: los códigos reemplazados, que de otro modo aparecerían como productos inactivos, integran su demanda a la del código vigente. Así, las series afectadas dejan de mostrar una caída artificial en el código anterior y un arranque desde cero en el nuevo. El control final reporta cuántas líneas de pedido se reasignaron y cuántas referencias quedan en el maestro después de homologar.

In [ ]:
# Bitácora de trazabilidad: registra las filas en cada paso de preparación
# (se completa en la sección 2 y alimenta el reporte de calidad)
TRAZABILIDAD = []
def registrar(fuente, paso, df):
    n = df.count()
    TRAZABILIDAD.append({"fuente": fuente, "paso": paso, "filas": n})
    return n

for nombre, df in [("Pedidos", df_pedidos), ("Ventas", df_ventas),
                   ("Devoluciones", df_devoluciones), ("Inventario", df_inventario)]:
    registrar(nombre, "0. Snapshot original", df)

# 1.1.b Actividad diaria por fuente
def actividad_diaria(df, col, nombre):
    return (df.groupBy(F.to_date(col).alias("fecha"))
              .agg(F.count("*").alias("lineas"))
              .withColumn("fuente", F.lit(nombre)))

pd_act = (
    actividad_diaria(df_pedidos, "DocDate", "Pedidos")
    .unionByName(actividad_diaria(df_ventas, "DocDate", "Ventas"))
    .unionByName(actividad_diaria(df_devoluciones, "DocDate", "Devoluciones"))
    .unionByName(actividad_diaria(df_inventario, "FechaCierre", "Inventario"))
    .toPandas()
)
pd_act["fecha"] = pd.to_datetime(pd_act["fecha"])
pd_act["mes"] = pd_act["fecha"].dt.to_period("M")

# 1.1.c Cobertura por fuente y meses completos
def primer_mes_completo(d):
    return d.to_period("M") if d.day == 1 else d.to_period("M") + 1

def ultimo_mes_completo(d):
    return d.to_period("M") if d.is_month_end else d.to_period("M") - 1

cobertura = pd_act.groupby("fuente").agg(
    fecha_min=("fecha", "min"), fecha_max=("fecha", "max"),
    dias_con_actividad=("fecha", "nunique"), lineas=("lineas", "sum"),
)
cobertura["dias_calendario"] = (cobertura["fecha_max"] - cobertura["fecha_min"]).dt.days + 1
cobertura["pct_dias_activos"] = cobertura["dias_con_actividad"] / cobertura["dias_calendario"] * 100
cobertura["primer_mes_completo"] = cobertura["fecha_min"].apply(primer_mes_completo)
cobertura["ultimo_mes_completo"] = cobertura["fecha_max"].apply(ultimo_mes_completo)
display(cobertura.reset_index().astype({"primer_mes_completo": str, "ultimo_mes_completo": str}))

# Extremos del histórico: primeros y últimos meses de cada fuente
mensual = (pd_act.groupby(["fuente", "mes"])
                 .agg(lineas=("lineas", "sum"), dias_activos=("fecha", "nunique"))
                 .reset_index())
extremos = pd.concat([mensual.groupby("fuente").head(3), mensual.groupby("fuente").tail(2)])
display(extremos.sort_values(["fuente", "mes"]).astype({"mes": str}))

# 1.1.d Ventana de análisis: intersección de meses completos de las fuentes documento
#       (el inventario no define la ventana: es un saldo acumulado que se
#        recorta solo por el final, ver nota abajo)
docs = cobertura.loc[["Pedidos", "Ventas", "Devoluciones"]]
MES_INICIO = docs["primer_mes_completo"].max()
MES_FIN    = docs["ultimo_mes_completo"].min()

MES_INICIO_MANUAL = None   # ej. pd.Period("2023-01", "M") si la gráfica muestra arranque atípico
if MES_INICIO_MANUAL is not None:
    MES_INICIO = MES_INICIO_MANUAL

FECHA_INICIO = MES_INICIO.start_time.strftime("%Y-%m-%d")
FECHA_FIN    = MES_FIN.end_time.strftime("%Y-%m-%d")
N_MESES      = (MES_FIN - MES_INICIO).n + 1
print(f"Ventana de análisis: {FECHA_INICIO} a {FECHA_FIN}  ({N_MESES} meses completos)")

# Gráfica: líneas mensuales por fuente, con los meses excluidos sombreados
fuentes_orden = ["Pedidos", "Ventas", "Devoluciones", "Inventario"]
fig, axes = plt.subplots(len(fuentes_orden), 1, figsize=(11, 9), sharex=True)
for ax, fuente in zip(axes, fuentes_orden):
    m = mensual[mensual["fuente"] == fuente]
    x = m["mes"].dt.to_timestamp()
    ax.plot(x, m["lineas"], marker="o", markersize=3)
    ax.axvspan(x.min(), MES_INICIO.start_time, color="lightcoral", alpha=0.25)
    ax.axvspan(MES_FIN.end_time, x.max() + pd.Timedelta(days=31), color="lightcoral", alpha=0.25)
    ax.set_title(fuente, fontsize=10, loc="left")
    ax.set_ylabel("Líneas / mes")
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(axes[-1].get_xticklabels(), rotation=45)
fig.suptitle("Cobertura mensual por fuente (zona sombreada: fuera de la ventana de análisis)")
plt.tight_layout()
plt.show()

# 1.1.e Aplicar la ventana
def en_ventana(df, col, desde=FECHA_INICIO, hasta=FECHA_FIN):
    cond = F.to_date(col) <= F.lit(hasta)
    if desde is not None:
        cond = cond & (F.to_date(col) >= F.lit(desde))
    return df.filter(cond)

df_pedidos      = en_ventana(df_pedidos, "DocDate").cache()
df_ventas       = en_ventana(df_ventas, "DocDate").cache()
df_devoluciones = en_ventana(df_devoluciones, "DocDate").cache()
df_inventario   = en_ventana(df_inventario, "FechaCierre", desde=None).cache()  # solo corte final

for nombre, df in [("Pedidos", df_pedidos), ("Ventas", df_ventas),
                   ("Devoluciones", df_devoluciones), ("Inventario", df_inventario)]:
    registrar(nombre, "1. Ventana de análisis", df)

display(pd.DataFrame(TRAZABILIDAD).pivot(index="fuente", columns="paso", values="filas").reset_index())

**Criterio de la ventana de análisis.** Los extremos del histórico rara vez son meses completos: al inicio suele aparecer una carga concentrada en una sola fecha (documentos y saldos migrados desde el sistema anterior) y al final el mes de la extracción queda incompleto. Por eso la ventana se define como la intersección de los meses completos de las tres fuentes documento (Pedidos, Ventas y Devoluciones), con la opción de fijar el inicio a mano (`MES_INICIO_MANUAL`) si la gráfica muestra un arranque atípico.

La cobertura diaria también describe el proceso comercial: si los pedidos se registran en una proporción de días mayor que las facturas, la facturación opera en días hábiles mientras la captura de pedidos se extiende a fines de semana y festivos.

Todas las fuentes documento se recortan a la ventana. El inventario se recorta solo por la fecha final, porque la reconstrucción del stock diario necesita el último saldo anterior al inicio de la ventana. La tabla de trazabilidad registra cuántas filas excluye cada paso.

### 1.2 Evolución mensual de la demanda y nivel de servicio.
Esta sección caracteriza el comportamiento agregado de la demanda dentro de la ventana de análisis. Para el propósito del proyecto se distinguen dos magnitudes. La demanda corresponde a las unidades pedidas por los clientes, variable objetivo del modelo predictivo (OE3). Las ventas corresponden a las unidades efectivamente facturadas, que pueden ser menores que lo pedido cuando no hay inventario disponible. La brecha entre ambas es, por tanto, un primer indicador del nivel de servicio.

El análisis se realiza sobre una vista preliminar que conserva solo los documentos de artículo (DocType = 'I') no cancelados (CANCELED = 'N'). La justificación y el efecto detallado de estas reglas se documentan en la sección 2 (calidad de datos). Aquí se cuantifica su peso para dimensionar su impacto sobre el análisis exploratorio.

In [ ]:
# ----------------------------------------------------------------------------
# 1.2 Evolución mensual de la demanda y nivel de servicio
# ----------------------------------------------------------------------------

# 1.2.a Composición de los documentos por tipo y estado de cancelación
#   DocType : I = artículo, S = servicio
#   CANCELED: N = vigente, Y = cancelado, C = documento de cancelación
def perfil_documentos(df, nombre):
    return (df.groupBy(F.lit(nombre).alias("fuente"), "DocType", "CANCELED")
              .agg(F.countDistinct("DocEntry").alias("documentos"),
                   F.count("*").alias("lineas"),
                   F.sum(F.col("Quantity").cast("double")).alias("unidades"),
                   F.sum(F.col("LineTotal").cast("double")).alias("valor")))

pd_perfil = (perfil_documentos(df_pedidos, "Pedidos")
             .unionByName(perfil_documentos(df_ventas, "Ventas"))
             .unionByName(perfil_documentos(df_devoluciones, "Devoluciones"))
             .toPandas())
for c in ["lineas", "unidades", "valor"]:
    pd_perfil[f"pct_{c}"] = pd_perfil[c] / pd_perfil.groupby("fuente")[c].transform("sum") * 100
display(pd_perfil.sort_values(["fuente", "lineas"], ascending=[True, False]).round(2))

def vista_valida(df):
    return df.filter((F.col("DocType") == "I") & (F.col("CANCELED") == "N"))

ped_v, ven_v, dev_v = vista_valida(df_pedidos), vista_valida(df_ventas), vista_valida(df_devoluciones)

# 1.2.b Serie mensual agregada
def serie_mensual(df, sufijo):
    return (df.groupBy(F.trunc("DocDate", "MM").alias("mes"))
              .agg(F.sum(F.col("Quantity").cast("double")).alias(f"unidades_{sufijo}"),
                   F.sum(F.col("LineTotal").cast("double")).alias(f"valor_{sufijo}"),
                   F.countDistinct("DocEntry").alias(f"docs_{sufijo}")))

pd_mes = (serie_mensual(ped_v, "pedidas")
          .join(serie_mensual(ven_v, "facturadas"), "mes", "full")
          .join(serie_mensual(dev_v, "devueltas"), "mes", "full")
          .orderBy("mes").toPandas().fillna(0))
pd_mes["mes"] = pd.to_datetime(pd_mes["mes"])
pd_mes["ratio_fact_ped"]  = pd_mes["unidades_facturadas"] / pd_mes["unidades_pedidas"] * 100
pd_mes["tasa_devolucion"] = pd_mes["unidades_devueltas"] / pd_mes["unidades_facturadas"] * 100

# Crecimiento anual (solo años calendario completos dentro de la ventana)
pd_mes["anio"] = pd_mes["mes"].dt.year
ANIOS_COMPLETOS = [a for a in range(MES_INICIO.year, MES_FIN.year + 1)
                   if pd.Period(f"{a}-01", "M") >= MES_INICIO and pd.Period(f"{a}-12", "M") <= MES_FIN]
anual = (pd_mes[pd_mes["anio"].isin(ANIOS_COMPLETOS)]
         .groupby("anio")[["unidades_pedidas", "unidades_facturadas", "valor_facturadas"]].sum())
anual["crec_pedidas_%"] = anual["unidades_pedidas"].pct_change() * 100
display(anual.reset_index().round(1))

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
ax = axes[0]
ax.plot(pd_mes["mes"], pd_mes["unidades_pedidas"], marker="o", ms=3, label="Pedidas (demanda)")
ax.plot(pd_mes["mes"], pd_mes["unidades_facturadas"], marker="o", ms=3, label="Facturadas (ventas)")
ax2 = ax.twinx()
ax2.plot(pd_mes["mes"], pd_mes["unidades_devueltas"], color="gray", ls="--", lw=1, label="Devueltas (eje der.)")
ax2.set_ylabel("Unidades devueltas"); ax2.grid(False)
ax.set_ylabel("Unidades"); ax.set_title("Unidades mensuales por tipo de movimiento")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left")

ax = axes[1]
ax.plot(pd_mes["mes"], pd_mes["ratio_fact_ped"], marker="o", ms=3, label="Facturado / Pedido (%)")
ax.plot(pd_mes["mes"], pd_mes["tasa_devolucion"], marker="o", ms=3, label="Devuelto / Facturado (%)")
ax.axhline(100, color="gray", ls=":", lw=1)
ax.set_ylabel("%"); ax.set_title("Relación facturado/pedido y tasa de devolución"); ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.get_xticklabels(), rotation=45)
plt.tight_layout(); plt.show()

# 1.2.c Índice estacional de la demanda (descomposición clásica, media móvil 2x12)
pd_mes["tendencia"] = pd_mes["unidades_pedidas"].rolling(12, center=True).mean().rolling(2).mean().shift(-1)
pd_mes["indice_estacional"] = pd_mes["unidades_pedidas"] / pd_mes["tendencia"]
indice = (pd_mes.dropna(subset=["indice_estacional"])
                .groupby(pd_mes["mes"].dt.month)["indice_estacional"]
                .agg(["mean", "min", "max", "count"]))
indice["mean"] = indice["mean"] / indice["mean"].mean()   # normalizar a promedio 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(indice.index, indice["mean"], color="steelblue", alpha=0.8)
ax.errorbar(indice.index, indice["mean"],
            yerr=[indice["mean"] - indice["min"], indice["max"] - indice["mean"]],
            fmt="none", ecolor="gray", capsize=3)
ax.axhline(1, color="firebrick", ls="--", lw=1)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"])
ax.set_ylabel("Índice (1 = mes promedio)")
ax.set_title("Índice estacional de unidades pedidas (sin efecto de tendencia)")
plt.tight_layout(); plt.show()
display(indice.reset_index().rename(columns={"mes": "mes_calendario"}).round(3))

# 1.2.d Trazabilidad: ¿desde qué documento se generan las facturas y devoluciones?
ORIGENES = {17: "Pedido de cliente (ORDR)", 15: "Entrega (ODLN)", 13: "Factura (OINV)",
            16: "Devolución mercancía (ORDN)", 234000031: "Solicitud de devolución (ORRR)",
            203: "Anticipo (ODPI)", -1: "Sin documento base"}

def dist_base(df, nombre):
    return df.groupBy(F.lit(nombre).alias("fuente"), "BaseType").agg(F.count("*").alias("lineas"))

pd_base = dist_base(ven_v, "Ventas").unionByName(dist_base(dev_v, "Devoluciones")).toPandas()
pd_base["origen"] = pd_base["BaseType"].map(ORIGENES).fillna("Otro")
pd_base["pct"] = pd_base["lineas"] / pd_base.groupby("fuente")["lineas"].transform("sum") * 100
display(pd_base.sort_values(["fuente", "lineas"], ascending=[True, False]).round(2))

**Cómo leer los resultados de esta sección.**

- *Composición de los documentos.* Los documentos de servicio (`DocType = 'S'`) y los cancelados deberían tener un peso marginal en Pedidos y Ventas. En Devoluciones, en cambio, los documentos de servicio suelen ser notas crédito de valor (descuentos o ajustes comerciales) sin unidades asociadas, que no deben computarse como devolución física. Además, en SAP la cancelación de una factura genera un documento compensatorio (`CANCELED = 'C'`) con las mismas unidades del documento cancelado (`CANCELED = 'Y'`): sumar ambos sin filtrar duplicaría el movimiento. En Pedidos la cancelación no genera documento compensatorio, así que basta con excluir los cancelados.
- *Tendencia.* El crecimiento anual de las unidades pedidas y del valor facturado indica si el modelo predictivo debe capturar una tendencia explícita. Si el valor crece más rápido que las unidades, también hay un efecto de precio.
- *Relación facturado/pedido.* Una relación sistemáticamente menor a 100 % indica demanda no atendida (quiebres de inventario o pedidos cerrados sin despacho). Los valores mensuales por encima de 100 % se explican por el desfase entre la fecha del pedido y la de la factura, por lo que la relación mensual es solo una aproximación; el nivel de servicio se mide con precisión a nivel de línea de pedido con la cantidad entregada (sección 3.1).
- *Devoluciones y trazabilidad.* Si buena parte de las devoluciones no tiene documento base, no pueden trazarse de forma sistemática a la factura de origen y se atribuyen a nivel de producto y mes. La distribución de `BaseType` en Ventas confirma si el flujo comercial sigue la secuencia pedido → entrega → factura.
- *Estacionalidad.* El índice estacional se calcula después de remover la tendencia con una media móvil centrada de 2×12 meses. Con pocos años de historia cada índice se estima con dos o tres observaciones por mes, así que debe tomarse como indicativo. Parte del patrón puede deberse a la política de precios y no al consumo (sección 3.2).

### 1.3 Dimensionamiento del portafolio y ciclo de vida de los productos.
El maestro de artículos incluye todos los productos terminados vendibles registrados en el ERP, pero no todos tienen demanda activa. Algunos no registran pedidos en la ventana de análisis, otros dejaron de pedirse (descontinuados o en agotamiento de línea) y otros son lanzamientos recientes con poca historia. A la inversa, en los pedidos aparecen códigos que no pertenecen al maestro de producto terminado. Esta sección cruza el maestro con la demanda observada para delimitar el universo de productos sobre el cual se construirá la clasificación (Fase 2), y asigna a cada producto una etapa de ciclo de vida según su actividad en la ventana.

In [ ]:
# ----------------------------------------------------------------------------
# 1.3 Dimensionamiento del portafolio y ciclo de vida de los SKU
# ----------------------------------------------------------------------------
MESES_INACTIVO = 6    # sin pedidos en los últimos N meses de la ventana -> inactivo
MESES_NUEVO    = 12   # primer pedido dentro de los últimos N meses -> nuevo

CORTE_INACTIVO = (MES_FIN - MESES_INACTIVO + 1).start_time
CORTE_NUEVO    = (MES_FIN - MESES_NUEVO + 1).start_time
print(f"Inactivo: último pedido antes de {CORTE_INACTIVO:%Y-%m-%d}")
print(f"Nuevo:    primer pedido desde   {CORTE_NUEVO:%Y-%m-%d}")

# 1.3.a Demanda observada por SKU (vista válida, dentro de la ventana)
sku_ped = (ped_v.filter(F.col("ItemCode").isNotNull())
           .groupBy("ItemCode")
           .agg(F.min("DocDate").alias("primer_pedido"),
                F.max("DocDate").alias("ultimo_pedido"),
                F.countDistinct(F.trunc("DocDate", "MM")).alias("meses_con_pedido"),
                F.countDistinct("DocEntry").alias("num_pedidos"),
                F.countDistinct("CardCode").alias("num_clientes"),
                F.sum(F.col("Quantity").cast("double")).alias("unidades_pedidas"),
                F.sum(F.col("LineTotal").cast("double")).alias("valor_pedido")))

# Nombres de respaldo para códigos fuera del maestro (Inventario trae ItemName de OITM)
nombres_inv = (df_inventario.select("ItemCode", F.col("ItemName").alias("ItemName_inv"))
                            .dropDuplicates(["ItemCode"]))

maestro = (df_productos.select("ItemCode", "ItemName", "ItmsGrpNam", "validFor",
                               "CreateDate", F.col("OnHand").cast("double").alias("OnHand"),
                               F.col("AvgPrice").cast("double").alias("AvgPrice"))
                       .withColumn("en_maestro", F.lit(True)))

pdu = (maestro.join(sku_ped, "ItemCode", "full")
              .join(nombres_inv, "ItemCode", "left")
              .withColumn("en_maestro", F.coalesce("en_maestro", F.lit(False)))
              .withColumn("ItemName", F.coalesce("ItemName", "ItemName_inv"))
              .drop("ItemName_inv")
              .toPandas())

# 1.3.b Etapa de ciclo de vida
def ciclo_vida(r):
    if not r["en_maestro"]:
        return "5. Fuera del maestro PT"
    if pd.isna(r["ultimo_pedido"]):
        return "4. Sin pedidos en la ventana"
    if r["ultimo_pedido"] < CORTE_INACTIVO:
        return f"3. Inactivo (sin pedidos últimos {MESES_INACTIVO} meses)"
    if r["primer_pedido"] >= CORTE_NUEVO:
        return f"2. Nuevo (< {MESES_NUEVO} meses de historia)"
    return "1. Activo con historia"

pdu["ciclo_vida"] = pdu.apply(ciclo_vida, axis=1)

resumen_ciclo = (pdu.groupby("ciclo_vida")
                    .agg(skus=("ItemCode", "count"),
                         unidades=("unidades_pedidas", "sum"),
                         valor=("valor_pedido", "sum"))
                    .sort_index())
for c in ["skus", "unidades", "valor"]:
    resumen_ciclo[f"pct_{c}"] = resumen_ciclo[c] / resumen_ciclo[c].sum() * 100
display(resumen_ciclo.reset_index().round(2))

print(f"SKUs en maestro PT:               {pdu['en_maestro'].sum()}")
print(f"SKUs del maestro con pedidos:     {(pdu['en_maestro'] & pdu['ultimo_pedido'].notna()).sum()}")
print(f"Códigos con pedidos fuera maestro:{(~pdu['en_maestro']).sum()}")
print(f"Clientes distintos con pedidos:   {ped_v.select('CardCode').distinct().count()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
etiquetas = resumen_ciclo.index
axes[0].barh(etiquetas, resumen_ciclo["pct_skus"], color="steelblue")
axes[0].set_title("% de SKUs"); axes[0].invert_yaxis()
axes[1].barh(etiquetas, resumen_ciclo["pct_unidades"], color="darkorange")
axes[1].set_title("% de unidades pedidas")
for ax, col in zip(axes, ["pct_skus", "pct_unidades"]):
    for i, v in enumerate(resumen_ciclo[col]):
        ax.text(v, i, f" {v:.1f}%", va="center", fontsize=8)
fig.suptitle("Portafolio por etapa de ciclo de vida (ventana de análisis)")
plt.tight_layout(); plt.show()

# 1.3.c Detalle: códigos fuera del maestro PT (esperado: material promocional)
display(pdu[~pdu["en_maestro"]]
        .sort_values("unidades_pedidas", ascending=False)
        [["ItemCode", "ItemName", "num_pedidos", "unidades_pedidas", "valor_pedido",
          "primer_pedido", "ultimo_pedido"]])

# 1.3.d Detalle: SKUs sin pedidos, por vigencia y existencia en bodega
sin_ped = pdu[pdu["ciclo_vida"].str.startswith("4.")].copy()
sin_ped["con_stock"] = sin_ped["OnHand"] > 0
display(sin_ped.groupby(["validFor", "con_stock"])
               .agg(skus=("ItemCode", "count"), unidades_en_mano=("OnHand", "sum"))
               .reset_index())

# 1.3.e Muestras gratuitas dentro del maestro PT y universo de clasificación
#       Criterio: descripción que coincide con PATRON_NOMBRE_MUESTRA, código con PATRON_CODIGO_MUESTRA
#       (cuando no registra venta real) o >= 90 % de sus líneas de pedido a precio cero
precio_cero = (ped_v.groupBy("ItemCode")
                    .agg(F.avg((F.col("Price") == 0).cast("int")).alias("pct_lineas_precio_cero"))
                    .toPandas())
pdu = pdu.drop(columns=["pct_lineas_precio_cero"], errors="ignore").merge(precio_cero, on="ItemCode", how="left")
pdu["es_muestra"] = pdu["en_maestro"] & (
    pdu["ItemName"].str.upper().str.contains(PATRON_NOMBRE_MUESTRA, na=False)
    | (pdu["ItemCode"].str.upper().str.contains(PATRON_CODIGO_MUESTRA, na=False)
       & (pdu["unidades_pedidas"].isna() | (pdu["pct_lineas_precio_cero"] >= 0.5)))
    | (pdu["pct_lineas_precio_cero"] >= 0.9)
)

display(pdu[pdu["es_muestra"]]
        .groupby("ciclo_vida")
        .agg(skus=("ItemCode", "count"), unidades_pedidas=("unidades_pedidas", "sum"),
             unidades_en_mano=("OnHand", "sum"))
        .reset_index())

# Muestras que estaban dentro del universo (revisión manual)
display(pdu[pdu["es_muestra"] & pdu["ciclo_vida"].str.startswith(("1.", "2."))]
        [["ItemCode", "ItemName", "ciclo_vida", "unidades_pedidas", "valor_pedido", "pct_lineas_precio_cero"]]
        .sort_values("unidades_pedidas", ascending=False))

# Universo de clasificación: activos + nuevos, sin muestras gratuitas
pdu["en_alcance"]     = pdu["ciclo_vida"].str.startswith(("1.", "2.")) & ~pdu["es_muestra"]
pdu["historia_corta"] = pdu["ciclo_vida"].str.startswith("2.")
df_sku_universo = spark.createDataFrame(
    pdu[["ItemCode", "ItemName", "ItmsGrpNam", "ciclo_vida", "es_muestra", "en_alcance", "historia_corta"]]
)
print(f"SKUs en el universo de clasificación: {pdu['en_alcance'].sum()} "
      f"({pdu.loc[pdu['en_alcance'], 'unidades_pedidas'].sum() / pdu['unidades_pedidas'].sum() * 100:.1f}% de las unidades)")


# 1.3.f Perfil de líneas comerciales en el universo
pdu = pdu.drop(columns=["NombreLinea"], errors="ignore").merge(
    df_productos.select("ItemCode", "NombreLinea").toPandas(), on="ItemCode", how="left")
u = pdu[pdu["en_alcance"]]
perfil_lineas = (u.groupby("NombreLinea", dropna=False)
                  .agg(skus=("ItemCode", "count"), unidades=("unidades_pedidas", "sum"),
                       valor=("valor_pedido", "sum"), nuevos=("historia_corta", "sum"))
                  .sort_values("valor", ascending=False))
perfil_lineas["pct_valor"] = perfil_lineas["valor"] / perfil_lineas["valor"].sum() * 100
perfil_lineas["pct_valor_acum"] = perfil_lineas["pct_valor"].cumsum()
print(f"Líneas en el universo: {u['NombreLinea'].nunique()} | SKUs sin línea: {u['NombreLinea'].isna().sum()}")
display(perfil_lineas.reset_index().round(2))

**Cómo leer los resultados.** Cada producto del maestro (ya homologado) recibe una etapa de ciclo de vida según dos umbrales: inactivo si no registra pedidos en los últimos `MESES_INACTIVO` meses de la ventana, y nuevo si su primer pedido cae dentro de los últimos `MESES_NUEVO` meses. En portafolios amplios lo habitual es una fuerte asimetría: una parte del catálogo concentra casi toda la demanda, mientras que otra arrastra referencias descontinuadas o en agotamiento que no deben pesar en la priorización.

Las tablas de detalle permiten revisar tres casos particulares:

- Los códigos con pedidos que no pertenecen al maestro de producto terminado (material promocional, logístico o cobros de servicio).
- Los productos sin pedidos en la ventana, separados por vigencia y existencias: muestras, lanzamientos recientes que aún no tienen su primer pedido o stock inmovilizado con riesgo de vencimiento.
- Las muestras gratuitas. Pueden estar registradas en el maestro como producto terminado vendible, por lo que los filtros de la extracción no las excluyen; aquí se identifican por su denominación, por su codificación o por tener la mayoría de sus líneas a precio cero, y se excluyen del universo como control para futuras actualizaciones de los datos.

Las existencias del maestro (`OnHand`) son una fotografía al momento de la extracción y no a la fecha de cierre de la ventana. El portafolio activo (activos con historia y nuevos, sin muestras) es la base del análisis exploratorio de las secciones 1.4 a 1.8; la delimitación final del universo se presenta en la sección 1.9.

### 1.4 Concentración de la demanda.
El análisis de Pareto permite dimensionar qué tan concentrada está la demanda en un subconjunto reducido de productos y constituye la base del método de clasificación ABC. Se aplica sobre el universo de clasificación definido en la sección anterior, con dos criterios (unidades pedidas y valor pedido) y dos periodos de referencia (la ventana completa y los últimos doce meses). La comparación permite evaluar si la priorización de productos es robusta o si depende del criterio y del horizonte elegidos, lo cual es determinante para el diseño de la clasificación en la Fase 2.

En esta fase, los grupos A, B y C derivados del análisis de Pareto se utilizan únicamente con fines exploratorios, como una forma de describir el comportamiento de los productos según su importancia. La clasificación del portafolio, que constituye el objetivo de la Fase 2, se construirá a partir de las variables continuas de la tabla de características; los métodos ABC, XYZ y de patrones de demanda se emplearán en esa fase como referencias de comparación.

In [ ]:
# ----------------------------------------------------------------------------
# 1.4 Concentración de la demanda (Pareto / ABC exploratorio)
# ----------------------------------------------------------------------------
INICIO_12M = (MES_FIN - 11).start_time.strftime("%Y-%m-%d")
PERIODOS = {"Ventana completa": FECHA_INICIO, "Últimos 12 meses": INICIO_12M}
CRITERIOS = {"unidades": "Unidades pedidas", "valor": "Valor pedido"}

universo_ids = df_sku_universo.filter("en_alcance").select("ItemCode")

def agregado_sku(desde):
    return (ped_v.join(universo_ids, "ItemCode", "left_semi")
                 .filter(F.to_date("DocDate") >= F.lit(desde))
                 .groupBy("ItemCode")
                 .agg(F.sum(F.col("Quantity").cast("double")).alias("unidades"),
                      F.sum(F.col("LineTotal").cast("double")).alias("valor"))
                 .toPandas())

def curva_pareto(pdf, col):
    s = pdf[["ItemCode", col]].sort_values(col, ascending=False).reset_index(drop=True)
    s["pct_acum"] = s[col].cumsum() / s[col].sum() * 100
    s["pct_skus"] = (s.index + 1) / len(s) * 100
    previo = s["pct_acum"].shift(fill_value=0)          # el SKU que cruza el umbral queda en la clase superior
    s["abc"] = np.select([previo < 80, previo < 95], ["A", "B"], default="C")
    return s

curvas, filas = {}, []
for periodo, desde in PERIODOS.items():
    base = agregado_sku(desde)
    for col, nombre in CRITERIOS.items():
        c = curva_pareto(base, col)
        curvas[(periodo, col)] = c
        conteo = c["abc"].value_counts()
        filas.append({"periodo": periodo, "criterio": nombre, "skus_total": len(c),
                      **{f"skus_{k}": int(conteo.get(k, 0)) for k in "ABC"},
                      "pct_skus_A": conteo.get("A", 0) / len(c) * 100,
                      "pct_skus_para_50": (c["pct_acum"] <= 50).sum() / len(c) * 100})
pd_pareto = pd.DataFrame(filas)
display(pd_pareto.round(1))

# Curvas de Pareto
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (col, nombre) in zip(axes, CRITERIOS.items()):
    for periodo, estilo in zip(PERIODOS, ["-", "--"]):
        c = curvas[(periodo, col)]
        ax.plot(c["pct_skus"], c["pct_acum"], ls=estilo, lw=2, label=periodo)
    for umbral in (80, 95):
        ax.axhline(umbral, color="gray", ls=":", lw=1)
    ax.set_title(f"Curva de Pareto — {nombre}")
    ax.set_xlabel("% acumulado de SKUs"); ax.legend(loc="lower right")
axes[0].set_ylabel("% acumulado de la demanda")
plt.tight_layout(); plt.show()

# 1.4.b Coincidencia entre criterios: ABC por unidades vs ABC por valor (últimos 12 meses)
abc_12m = (curvas[("Últimos 12 meses", "unidades")][["ItemCode", "abc"]]
           .merge(curvas[("Últimos 12 meses", "valor")][["ItemCode", "abc"]],
                  on="ItemCode", suffixes=("_unidades", "_valor")))
ct_criterio = pd.crosstab(abc_12m["abc_unidades"], abc_12m["abc_valor"], margins=True)
display(ct_criterio.reset_index())
print(f"Coincidencia unidades vs valor: {(abc_12m['abc_unidades'] == abc_12m['abc_valor']).mean() * 100:.1f}%")

# 1.4.c Estabilidad temporal: ABC por valor, ventana completa vs últimos 12 meses
abc_t = (curvas[("Ventana completa", "valor")][["ItemCode", "abc"]]
         .merge(curvas[("Últimos 12 meses", "valor")][["ItemCode", "abc"]],
                on="ItemCode", how="outer", suffixes=("_ventana", "_12m"))
         .fillna("Sin demanda"))
ct_tiempo = pd.crosstab(abc_t["abc_ventana"], abc_t["abc_12m"], margins=True)
display(ct_tiempo.reset_index())
print(f"Coincidencia ventana vs 12 meses (valor): {(abc_t['abc_ventana'] == abc_t['abc_12m']).mean() * 100:.1f}%")

**Cómo leer los resultados.** Las curvas de Pareto muestran qué proporción de productos concentra el 80 % y el 50 % de la demanda. Si las curvas de la ventana completa y de los últimos doce meses se superponen, el grado de concentración es una característica estable del negocio.

La estabilidad del agregado, sin embargo, no implica estabilidad en la asignación individual. Las tablas cruzadas comparan los grupos ABC por unidades frente a valor, y por ventana completa frente a los últimos doce meses. Los cambios de grupo suelen concentrarse en las fronteras del grupo B: productos de alta rotación y bajo precio (B en unidades y C en valor) y de baja rotación y alto precio (lo contrario).

De este análisis se derivan tres implicaciones para la Fase 2. Primero, una clasificación ABC de criterio único es sensible a la elección del criterio, lo que respalda un enfoque multicriterio (Purwandaru et al., 2024). Segundo, como la pertenencia a cada grupo evoluciona en el tiempo, la clasificación debe calcularse sobre un periodo reciente (los últimos doce meses). Tercero, y más importante, el volumen y el valor indican qué productos pesan más para el negocio, pero no cuáles son más difíciles de pronosticar: dos productos de alto valor pueden tener patrones muy distintos, uno estable y otro errático, y solo el segundo exige un esfuerzo predictivo mayor. Por eso la clasificación debe incorporar también la variabilidad y la intermitencia de la demanda, que se caracterizan en la sección siguiente.

### 1.5 Perfil de la demanda por producto: frecuencia, variabilidad e intermitencia.
El volumen y el valor indican qué productos son más importantes para el negocio, pero no qué tan difícil es pronosticar su demanda. Para caracterizar esta segunda dimensión se calcularon, para cada producto del universo, dos indicadores propuestos por Syntetos, Boylan y Croston (2005): el intervalo promedio entre demandas (ADI), que mide con qué frecuencia se presentan periodos sin demanda, y el cuadrado del coeficiente de variación del tamaño de la demanda (CV²), que mide qué tan variable es la cantidad pedida cuando la demanda ocurre. Con los umbrales de referencia (ADI = 1,32 y CV² = 0,49), cada serie se clasifica en uno de cuatro patrones: suave, errática, intermitente o irregular. Complementariamente, se calculó el coeficiente de variación de la serie mensual de los últimos doce meses, base de la clasificación XYZ. Los indicadores se estimaron a nivel mensual y semanal. Para cada producto se considera su historia desde el primer periodo con demanda dentro de la ventana, de modo que los productos nuevos no se penalicen por los periodos anteriores a su lanzamiento.

In [ ]:
# ----------------------------------------------------------------------------
# 1.5 Perfil de demanda por SKU: frecuencia, variabilidad e intermitencia
# ----------------------------------------------------------------------------
ADI_CORTE, CV2_CORTE = 1.32, 0.49        # Syntetos, Boylan & Croston (2005)
XYZ_CORTES = (0.5, 1.0)                  # CV <= 0.5 -> X, <= 1.0 -> Y, > 1.0 -> Z

ped_u = ped_v.join(universo_ids, "ItemCode", "left_semi")

# 1.5.a Series densas (con ceros) por SKU, recortadas desde su primer periodo con demanda
def densificar(pdf, periodos):
    idx = pd.MultiIndex.from_product([pdf["ItemCode"].unique(), periodos],
                                     names=["ItemCode", "periodo"])
    d = (pdf.set_index(["ItemCode", "periodo"])["unidades"]
            .reindex(idx, fill_value=0).reset_index())
    primero = d[d["unidades"] > 0].groupby("ItemCode")["periodo"].min().rename("inicio")
    d = d.merge(primero, on="ItemCode")
    return d[d["periodo"] >= d["inicio"]].drop(columns="inicio")

# Mensual
pd_m = (ped_u.groupBy("ItemCode", F.trunc("DocDate", "MM").alias("periodo"))
             .agg(F.sum(F.col("Quantity").cast("double")).alias("unidades")).toPandas())
pd_m["periodo"] = pd.to_datetime(pd_m["periodo"])
meses = pd.date_range(MES_INICIO.start_time, MES_FIN.start_time, freq="MS")
serie_mes = densificar(pd_m, meses)

# Semanal (semanas lunes-domingo completamente dentro de la ventana)
pd_w = (ped_u.groupBy("ItemCode", F.date_trunc("week", "DocDate").alias("periodo"))
             .agg(F.sum(F.col("Quantity").cast("double")).alias("unidades")).toPandas())
pd_w["periodo"] = pd.to_datetime(pd_w["periodo"])
inicio = pd.Timestamp(FECHA_INICIO)
primer_lunes = inicio if inicio.weekday() == 0 else inicio + pd.offsets.Week(weekday=0)
ultimo_lunes = pd.Timestamp(FECHA_FIN) - pd.Timedelta(days=6)
ultimo_lunes = ultimo_lunes - pd.Timedelta(days=ultimo_lunes.weekday())
semanas = pd.date_range(primer_lunes, ultimo_lunes, freq="W-MON")
serie_sem = densificar(pd_w[pd_w["periodo"].isin(semanas)], semanas)
print(f"Periodos: {len(meses)} meses, {len(semanas)} semanas")

# 1.5.b ADI, CV² y patrón SBC
def perfil_sbc(serie, sufijo):
    def calc(g):
        y = g["unidades"].to_numpy()
        nz = y[y > 0]
        n, k = len(y), len(nz)
        return pd.Series({
            f"periodos_{sufijo}": n,
            f"periodos_con_demanda_{sufijo}": k,
            f"pct_periodos_con_demanda_{sufijo}": k / n * 100,
            f"adi_{sufijo}": n / k if k else np.nan,
            f"cv2_{sufijo}": (nz.std(ddof=0) / nz.mean()) ** 2 if k > 1 else np.nan,
        })
    out = serie.groupby("ItemCode").apply(calc).reset_index()
    adi, cv2 = out[f"adi_{sufijo}"], out[f"cv2_{sufijo}"]
    out[f"sbc_{sufijo}"] = np.select(
        [(adi < ADI_CORTE) & (cv2 < CV2_CORTE), (adi < ADI_CORTE) & (cv2 >= CV2_CORTE),
         (adi >= ADI_CORTE) & (cv2 < CV2_CORTE), (adi >= ADI_CORTE) & (cv2 >= CV2_CORTE)],
        ["Suave", "Errática", "Intermitente", "Irregular"], default="Sin datos")
    return out

perfil = perfil_sbc(serie_mes, "mes").merge(perfil_sbc(serie_sem, "sem"), on="ItemCode")

# 1.5.c CV mensual de los últimos 12 meses y clase XYZ (requiere 12 meses de historia)
ult12 = serie_mes[serie_mes["periodo"] >= pd.Timestamp(INICIO_12M)]
cv12 = (ult12.groupby("ItemCode")["unidades"]
             .agg(n="count", media="mean", desv=lambda s: s.std(ddof=0)).reset_index())
cv12["cv_12m"] = cv12["desv"] / cv12["media"]
cv12.loc[cv12["n"] < 12, "cv_12m"] = np.nan
perfil = perfil.merge(cv12[["ItemCode", "cv_12m"]], on="ItemCode", how="left")
perfil["xyz"] = np.select([perfil["cv_12m"] <= XYZ_CORTES[0], perfil["cv_12m"] <= XYZ_CORTES[1],
                           perfil["cv_12m"] > XYZ_CORTES[1]], ["X", "Y", "Z"], default="Historia < 12m")

# Unir ABC (últimos 12 meses) y marcas del universo
abc_ref = (curvas[("Últimos 12 meses", "valor")][["ItemCode", "abc", "valor"]]
           .rename(columns={"abc": "abc_valor"})
           .merge(curvas[("Últimos 12 meses", "unidades")][["ItemCode", "abc", "unidades"]]
                  .rename(columns={"abc": "abc_unidades"}), on="ItemCode"))
pd_perfil_sku = (perfil.merge(abc_ref, on="ItemCode", how="left")
                       .merge(pdu[["ItemCode", "ItemName", "historia_corta"]], on="ItemCode", how="left"))

# 1.5.d Distribución de patrones: mensual vs semanal
orden_sbc = ["Suave", "Errática", "Intermitente", "Irregular", "Sin datos"]
dist = pd.DataFrame({
    g: pd_perfil_sku.groupby(f"sbc_{g}").agg(skus=("ItemCode", "count"), unidades=("unidades", "sum"))
         .reindex(orden_sbc).fillna(0).assign(pct_skus=lambda d: d.skus / d.skus.sum() * 100,
                                              pct_unidades=lambda d: d.unidades / d.unidades.sum() * 100)
         .stack() for g in ["mes", "sem"]}).unstack()
display(dist.round(1).reset_index())
display(pd.crosstab(pd_perfil_sku["sbc_mes"], pd_perfil_sku["sbc_sem"], margins=True).reset_index())

# 1.5.e Mapa ADI vs CV² por granularidad, coloreado por clase ABC (valor)
colores = {"A": "firebrick", "B": "darkorange", "C": "steelblue"}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, g, titulo in zip(axes, ["mes", "sem"], ["Mensual", "Semanal"]):
    for clase, color in colores.items():
        sub = pd_perfil_sku[pd_perfil_sku["abc_valor"] == clase]
        ax.scatter(sub[f"adi_{g}"], sub[f"cv2_{g}"], s=18, alpha=0.7, color=color, label=f"Clase {clase}")
    ax.axvline(ADI_CORTE, color="gray", ls="--", lw=1); ax.axhline(CV2_CORTE, color="gray", ls="--", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("ADI (log)"); ax.set_ylabel("CV² (log)"); ax.set_title(f"Patrón de demanda — {titulo}")
    for x, y, t in [(0.02, 0.02, "Suave"), (0.02, 0.95, "Errática"), (0.75, 0.02, "Intermitente"), (0.75, 0.95, "Irregular")]:
        ax.text(x, y, t, transform=ax.transAxes, fontsize=9, color="dimgray",
                va="bottom" if y < 0.5 else "top")
    ax.legend(loc="center right", fontsize=8)
plt.tight_layout(); plt.show()

# 1.5.f Importancia vs pronosticabilidad: ABC x SBC (semanal) y ABC x XYZ
display(pd.crosstab(pd_perfil_sku["abc_valor"], pd_perfil_sku["sbc_sem"], margins=True).reset_index())
display(pd.crosstab(pd_perfil_sku["abc_valor"], pd_perfil_sku["xyz"], margins=True).reset_index())

# Productos A con patrón difícil: los candidatos naturales a "mayor precisión requerida"
display(pd_perfil_sku[(pd_perfil_sku["abc_valor"] == "A")
                      & pd_perfil_sku["sbc_sem"].isin(["Errática", "Irregular", "Intermitente"])]
        [["ItemCode", "ItemName", "abc_unidades", "adi_sem", "cv2_sem", "sbc_sem", "cv_12m", "xyz"]]
        .sort_values("cv2_sem", ascending=False).round(3))

**Cómo leer los resultados.**

- *Patrones de demanda.* La distribución de patrones SBC se compara entre la granularidad mensual y la semanal. La tabla de contingencia indica si el cambio de patrón al pasar a semanas se debe a mayor variabilidad (suave → errática) o a mayor intermitencia (suave → intermitente). Si predomina lo primero, la agregación semanal no introduce periodos sin demanda sino fluctuaciones en el tamaño de los pedidos que se compensan dentro del mes; eso favorece la granularidad mensual para el modelo predictivo, mientras que la semanal exigiría métodos especializados para demanda errática o intermitente (Croston, TSB) en más productos.
- *Importancia frente a pronosticabilidad.* Los cruces ABC × SBC y ABC × XYZ muestran que ambas dimensiones están relacionadas pero no son equivalentes. Los casos clave son los productos de alto valor con patrón difícil: combinan alto impacto en el negocio con una demanda difícil de anticipar, de modo que un error de pronóstico en ellos tiene el mayor costo potencial (vencimiento o agotado). Una clasificación basada solo en volumen o valor los trataría igual que a los productos estables de su mismo grupo. Este es el argumento empírico del enfoque híbrido: la priorización debe combinar la importancia del producto con la dificultad de pronosticar su demanda.
- *Sustitución entre presentaciones.* Si entre los productos de alto valor con patrón difícil aparecen varias presentaciones de un mismo producto que no están en la tabla de equivalencias, su volatilidad puede deberse a que la demanda se desplaza de una referencia a otra. Esta hipótesis se evalúa en la fase de modelado.
- *Limitaciones.* El coeficiente de variación de doce meses, base de la clase XYZ, incluye el efecto de la tendencia, por lo que un producto en crecimiento puede aparecer con variabilidad media sin ser volátil. Los productos con menos de doce meses de historia no reciben clase XYZ. Los indicadores continuos (ADI, CV² y CV) se incorporan a la tabla de características por producto (sección 3.3) como insumo de la Fase 2.

- Control de consistencia: verifica que el universo se mantenga completo entre las secciones 1.3 y 1.5.

In [ ]:
# Control de consistencia del universo
print(pdu["ciclo_vida"].value_counts().sort_index(), "\n")
print("Muestras en activos/nuevos:", (pdu["es_muestra"] & pdu["ciclo_vida"].str.startswith(("1.", "2."))).sum())
print("Universo (pdu):            ", pdu["en_alcance"].sum())
print("Universo (df_sku_universo):", df_sku_universo.filter("en_alcance").count())
print("SKUs con ABC 12m:          ", len(curvas[("Últimos 12 meses", "valor")]))
print("SKUs en pd_perfil_sku:     ", len(pd_perfil_sku))

### 1.6 Concentración de la demanda por cliente.
La distribuidora atiende una base amplia y heterogénea de clientes, que incluye cadenas comerciales, droguerías independientes y consumidores del canal digital. Esta heterogeneidad incide en el comportamiento de la demanda: un producto cuya demanda depende de pocos clientes de gran tamaño tiende a presentar fluctuaciones más pronunciadas, ya que la decisión de compra de un solo cliente altera de forma significativa el volumen del periodo. Esta sección caracteriza la concentración de la demanda en dos niveles. A nivel del negocio, se mide qué proporción del valor concentran los principales clientes y qué tan recurrente es la base de compradores. A nivel de producto, se mide qué tan dependiente es la demanda de cada referencia de su cliente principal, y se evalúa si esa dependencia se asocia con la variabilidad de su demanda. Por confidencialidad, los clientes se presentan de forma anonimizada.

In [ ]:
# ----------------------------------------------------------------------------
# 1.6 Concentración de la demanda por cliente (consolidado por NIT)
# ----------------------------------------------------------------------------
# 1.6.0 Consolidación de sedes: C0001-1, C0001-2 -> C0001 (el código de cliente es NIT + sufijo de sede)
ped_c = ped_u.withColumn("ClienteID", F.regexp_replace("CardCode", r"-[^-]+$", ""))

sedes = (ped_c.groupBy("ClienteID")
              .agg(F.countDistinct("CardCode").alias("sedes"),
                   F.countDistinct("CardName").alias("nombres_distintos"))
              .toPandas())
print(f"Códigos de cliente (sede):   {ped_c.select('CardCode').distinct().count():,}")
print(f"Clientes consolidados (NIT): {len(sedes):,}")
print(f"NIT con más de una sede:     {(sedes['sedes'] > 1).sum():,}  (máx. {sedes['sedes'].max()} sedes)")

# Control de sanidad: NIT cuyas sedes tienen nombres distintos (el detalle solo se muestra en revisión interna)
revisar = sedes[sedes["nombres_distintos"] > 1]["ClienteID"].tolist()
print(f"NIT con nombres distintos entre sedes: {len(revisar)}")
if revisar and REVISION_INTERNA:
    display(ped_c.filter(F.col("ClienteID").isin(revisar[:20]))
                 .select("ClienteID", "CardCode", "CardName").distinct()
                 .orderBy("ClienteID", "CardCode"))

# 1.6.a Perfil por cliente consolidado (ventana completa)
pd_cli = (ped_c.groupBy("ClienteID")
               .agg(F.sum(F.col("LineTotal").cast("double")).alias("valor"),
                    F.sum(F.col("Quantity").cast("double")).alias("unidades"),
                    F.countDistinct("DocEntry").alias("pedidos"),
                    F.countDistinct("CardCode").alias("sedes"),
                    F.countDistinct(F.trunc("DocDate", "MM")).alias("meses_activos"),
                    F.countDistinct("ItemCode").alias("skus"))
               .toPandas()
               .sort_values("valor", ascending=False).reset_index(drop=True))
pd_cli["ranking"] = pd_cli.index + 1
pd_cli["pct_acum_valor"] = pd_cli["valor"].cumsum() / pd_cli["valor"].sum() * 100
pd_cli["pct_clientes"] = pd_cli["ranking"] / len(pd_cli) * 100

n_cli = len(pd_cli)
total_valor = pd_cli["valor"].sum()
conc_cli = pd.DataFrame([{
    "clientes_total": n_cli,
    "pct_valor_top1": pd_cli["valor"].iloc[0] / total_valor * 100,
    "pct_valor_top10": pd_cli["valor"].iloc[:10].sum() / total_valor * 100,
    "pct_valor_top100": pd_cli["valor"].iloc[:100].sum() / total_valor * 100,
    "clientes_para_50pct": int((pd_cli["pct_acum_valor"] < 50).sum() + 1),
    "clientes_para_80pct": int((pd_cli["pct_acum_valor"] < 80).sum() + 1),
}])
conc_cli["pct_clientes_para_80"] = conc_cli["clientes_para_80pct"] / n_cli * 100
display(conc_cli.round(2))

# Revisión interna: top 15 de clientes consolidados con su nombre (desactivado en la versión pública)
if REVISION_INTERNA:
    nombre_cli = (ped_c.groupBy("ClienteID").agg(F.first("CardName").alias("CardName")).toPandas())
    display(pd_cli.head(15).merge(nombre_cli, on="ClienteID", how="left")
                  [["ranking", "CardName", "sedes", "valor", "pedidos", "meses_activos", "skus"]])

# 1.6.b Recurrencia: clientes y valor según número de pedidos
pd_cli["tramo_pedidos"] = pd.cut(pd_cli["pedidos"], bins=[0, 1, 5, 12, 24, 48, np.inf],
                                 labels=["1", "2-5", "6-12", "13-24", "25-48", ">48"])
recurrencia = (pd_cli.groupby("tramo_pedidos", observed=False)
                     .agg(clientes=("ClienteID", "count"), valor=("valor", "sum")))
recurrencia["pct_clientes"] = recurrencia["clientes"] / n_cli * 100
recurrencia["pct_valor"] = recurrencia["valor"] / recurrencia["valor"].sum() * 100
display(recurrencia.reset_index().round(1))

# 1.6.c Dependencia de cada SKU respecto de sus clientes consolidados (últimos 12 meses)
w_sku = Window.partitionBy("ItemCode")
pd_conc_sku = (ped_c.filter(F.to_date("DocDate") >= F.lit(INICIO_12M))
               .groupBy("ItemCode", "ClienteID")
               .agg(F.sum(F.col("Quantity").cast("double")).alias("u"))
               .withColumn("share", F.col("u") / F.sum("u").over(w_sku))
               .groupBy("ItemCode")
               .agg(F.count("*").alias("clientes_12m"),
                    F.max("share").alias("share_top1_cliente"),
                    F.sum(F.col("share") ** 2).alias("hhi_clientes"))
               .toPandas())

pd_perfil_sku = (pd_perfil_sku.drop(columns=["clientes_12m", "share_top1_cliente", "hhi_clientes"], errors="ignore")
                              .merge(pd_conc_sku, on="ItemCode", how="left"))

display(pd_perfil_sku.groupby("abc_valor")
        .agg(skus=("ItemCode", "count"),
             clientes_12m_mediana=("clientes_12m", "median"),
             share_top1_mediana=("share_top1_cliente", "median"),
             skus_top1_mayor_50pct=("share_top1_cliente", lambda s: (s > 0.5).sum()))
        .reset_index().round(3))

# 1.6.d Dependencia vs volatilidad: global y DENTRO de cada clase ABC (controla el efecto tamaño)
vars_dep, vars_vol = ["share_top1_cliente", "hhi_clientes", "clientes_12m"], ["cv2_sem", "cv_12m"]
filas_corr = []
for grupo, sub in [("Global", pd_perfil_sku)] + list(pd_perfil_sku.groupby("abc_valor")):
    c = sub[vars_dep + vars_vol].corr(method="spearman").loc[vars_dep, vars_vol]
    for dep in vars_dep:
        filas_corr.append({"grupo": grupo, "variable": dep, "n": len(sub),
                           **{f"rho_{v}": c.loc[dep, v] for v in vars_vol}})
display(pd.DataFrame(filas_corr).round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
ax.plot(pd_cli["pct_clientes"], pd_cli["pct_acum_valor"], lw=2)
ax.axhline(80, color="gray", ls=":", lw=1); ax.axhline(50, color="gray", ls=":", lw=1)
ax.set_xscale("log"); ax.set_xlabel("% acumulado de clientes (log)")
ax.set_ylabel("% acumulado del valor pedido"); ax.set_title("Concentración del valor por cliente (NIT)")

ax = axes[1]
for clase, color in colores.items():
    sub = pd_perfil_sku[pd_perfil_sku["abc_valor"] == clase]
    ax.scatter(sub["share_top1_cliente"] * 100, sub["cv2_sem"], s=18, alpha=0.7, color=color, label=f"Clase {clase}")
ax.axhline(CV2_CORTE, color="gray", ls="--", lw=1)
ax.set_yscale("log"); ax.set_xlabel("% de la demanda del SKU en su cliente principal (12m)")
ax.set_ylabel("CV² semanal (log)"); ax.set_title("Dependencia del cliente principal vs. volatilidad")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Cómo leer los resultados.**

- *Consolidación de clientes.* En el ERP cada sede o punto de entrega de un cliente se registra con un código propio, compuesto por el NIT y un sufijo de sede. Como las decisiones de compra de cadenas y cooperativas suelen centralizarse, el análisis se hace a nivel de entidad legal (NIT). La consolidación puede cambiar de forma sustancial la lectura de la concentración, porque un cliente grande fragmentado en varias sedes aparece repartido en el ranking.
- *Concentración y recurrencia.* La curva de concentración y la tabla por tramos de pedidos muestran si el negocio depende de un núcleo reducido de clientes recurrentes acompañado de una base amplia de compradores ocasionales de bajo impacto.
- *Dependencia de los productos.* Un número alto de clientes no implica baja dependencia: la participación del cliente principal y el índice de Herfindahl-Hirschman (HHI) miden qué tanto depende cada producto de pocos compradores.
- *Dependencia y volatilidad.* La correlación global entre el número de clientes y la variabilidad está influida por el tamaño del producto (los productos con pocos clientes suelen ser también los de menor volumen). Por eso se calcula también dentro de cada clase ABC: si entre productos de importancia comparable la dependencia del cliente principal se asocia positivamente con la variabilidad, la decisión de compra de un solo cliente altera de forma significativa el volumen del periodo.

Con base en lo anterior, la participación del cliente principal, el HHI y el número de clientes se incorporan a la tabla de características como criterios candidatos para la Fase 2 y como variables candidatas para el modelo predictivo. Los nombres de los clientes solo se muestran en modo de revisión interna (`REVISION_INTERNA = True`) y nunca se publican.

### 1.7 Inventario y devoluciones.
Esta sección caracteriza las dos fuentes que complementan la demanda: el inventario, necesario para identificar los periodos en que la demanda no pudo atenderse por falta de existencias, y las devoluciones, que permiten estimar la demanda neta de cada producto. Dado que la distribuidora opera con varias bodegas que cumplen funciones distintas, el primer paso es establecer el papel de cada una a partir de su participación en los pedidos, en las devoluciones y en las existencias, con el fin de determinar cuáles representan inventario disponible para la venta.

In [ ]:
# ----------------------------------------------------------------------------
# 1.7 Inventario y devoluciones
# ----------------------------------------------------------------------------
QTY = F.col("Quantity").cast("double")
dev_u = dev_v.join(universo_ids, "ItemCode", "left_semi")
ven_u = ven_v.join(universo_ids, "ItemCode", "left_semi")

# 1.7.a Papel de cada bodega: salida de pedidos, entrada de devoluciones y stock
w_ult = Window.partitionBy("ItemCode_orig", "Warehouse").orderBy(F.desc("FechaCierre"))
ult_cierre = (df_inventario.withColumn("rn", F.row_number().over(w_ult)).filter("rn = 1")
                           .join(universo_ids.withColumn("en_universo", F.lit(1)), "ItemCode", "left"))

stock_bodega = (ult_cierre.groupBy("Warehouse")
                .agg(F.sum(F.col("StockCierreDia").cast("double")).alias("stock_final_total"),
                     F.sum(F.when(F.col("en_universo") == 1, F.col("StockCierreDia").cast("double")))
                      .alias("stock_final_universo"),
                     F.sum((F.col("StockCierreDia") < 0).cast("int")).alias("saldos_negativos")))

mov_bodega = (df_inventario.groupBy("Warehouse", "WhsName")
              .agg(F.countDistinct("ItemCode").alias("skus_con_movimiento"),
                   F.min("FechaCierre").alias("primer_mov"),
                   F.max("FechaCierre").alias("ultimo_mov")))

ped_bodega = ped_u.groupBy(F.col("WhsCode").alias("Warehouse")).agg(F.sum(QTY).alias("unid_pedidas"))
dev_bodega = dev_u.groupBy(F.col("WhsCode").alias("Warehouse")).agg(F.sum(QTY).alias("unid_devueltas"))

pd_bodegas = (mov_bodega.join(stock_bodega, "Warehouse", "left")
                        .join(ped_bodega, "Warehouse", "full")
                        .join(dev_bodega, "Warehouse", "full")
                        .toPandas().fillna({"unid_pedidas": 0, "unid_devueltas": 0}))
for c in ["unid_pedidas", "unid_devueltas", "stock_final_universo"]:
    pd_bodegas[f"pct_{c}"] = pd_bodegas[c] / pd_bodegas[c].sum() * 100
display(pd_bodegas.sort_values("unid_pedidas", ascending=False).round(1))

# 1.7.b Tasa de devolución por SKU (últimos 12 meses)
def unidades_12m(df, alias):
    return (df.filter(F.to_date("DocDate") >= F.lit(INICIO_12M))
              .groupBy("ItemCode").agg(F.sum(QTY).alias(alias)))

pd_dev_sku = (unidades_12m(ven_u, "unid_facturadas_12m")
              .join(unidades_12m(dev_u, "unid_devueltas_12m"), "ItemCode", "full")
              .toPandas().fillna(0))
pd_dev_sku["tasa_devolucion_12m"] = np.where(
    pd_dev_sku["unid_facturadas_12m"] > 0,
    pd_dev_sku["unid_devueltas_12m"] / pd_dev_sku["unid_facturadas_12m"] * 100, np.nan)

pd_perfil_sku = (pd_perfil_sku.drop(columns=["unid_facturadas_12m", "unid_devueltas_12m", "tasa_devolucion_12m"],
                                    errors="ignore")
                              .merge(pd_dev_sku, on="ItemCode", how="left"))

tot = pd_perfil_sku[["unid_facturadas_12m", "unid_devueltas_12m"]].sum()
print(f"Tasa de devolución global del universo (12m): {tot['unid_devueltas_12m'] / tot['unid_facturadas_12m'] * 100:.2f}%")

display(pd_perfil_sku.groupby("abc_valor")
        .agg(skus=("ItemCode", "count"),
             tasa_mediana=("tasa_devolucion_12m", "median"),
             tasa_p90=("tasa_devolucion_12m", lambda s: s.quantile(0.9)),
             skus_tasa_mayor_5pct=("tasa_devolucion_12m", lambda s: (s > 5).sum()),
             skus_sin_devoluciones=("unid_devueltas_12m", lambda s: (s == 0).sum()))
        .reset_index().round(2))

display(pd_perfil_sku[pd_perfil_sku["unid_facturadas_12m"] >= 100]
        .sort_values("tasa_devolucion_12m", ascending=False)
        [["ItemCode", "ItemName", "abc_valor", "sbc_mes", "unid_facturadas_12m",
          "unid_devueltas_12m", "tasa_devolucion_12m"]].head(15).round(2))

fig, ax = plt.subplots(figsize=(8, 4))
for clase, color in colores.items():
    s = pd_perfil_sku.loc[pd_perfil_sku["abc_valor"] == clase, "tasa_devolucion_12m"].dropna()
    ax.hist(s.clip(upper=20), bins=40, alpha=0.5, color=color, label=f"Clase {clase}")
ax.set_xlabel("Tasa de devolución 12m (%) — recortada en 20%")
ax.set_ylabel("Número de SKUs"); ax.set_title("Distribución de la tasa de devolución por clase ABC")
ax.legend(); plt.tight_layout(); plt.show()

#### Papel de las bodegas

La tabla de bodegas cruza, para cada una, su participación en las unidades pedidas, en las unidades devueltas y en las existencias. Con ella se distinguen la bodega principal de despacho, las bodegas de reserva de producto liberado por calidad, las de devoluciones, las de consignación ubicadas en instalaciones de clientes y las de uso interno (planeación de producción, acondicionamiento o material publicitario), que no participan en la venta. A partir de esta caracterización, y de la validación con el área de operaciones, se definen en la celda de parámetros las bodegas de inventario disponible para la venta (`BODEGAS_VENTA`) y la bodega de producto vencido (`BODEGA_VENCIDOS`). Las bodegas de consignación se excluyen del cálculo de disponibilidad porque su inventario está en poder del cliente, lo cual es una limitación del análisis de quiebres para ese canal. El control de saldos negativos verifica la consistencia del inventario reconstruido.

#### Devoluciones

La tasa de devolución se resume por clase ABC con la mediana, el percentil 90 y el número de productos por encima del 5 %. Si la dispersión aumenta hacia los productos de menor valor, o si las tasas más altas se concentran en una familia de productos, en esos casos los pedidos sobrestiman el consumo real del mercado.

En consecuencia, la variable objetivo del modelo predictivo se mantiene como la demanda bruta (unidades pedidas), coherente con el objetivo de anticipar los pedidos futuros. La tasa de devolución se incorpora a la tabla de características como criterio de clasificación, porque un producto con alta devolución implica un riesgo adicional de sobreinventario, y la demanda neta se conserva en la tabla mensual para análisis de sensibilidad.

#### Disponibilidad de inventario y obsolescencia.

Los pedidos registran la demanda de los clientes, pero su atención depende de que existan unidades disponibles. Para identificar los periodos de desabastecimiento se reconstruyó el stock diario de cada producto en las bodegas de venta. Dado que el ERP solo registra el saldo en los días con movimiento, el saldo de los días sin movimiento se completó con el último cierre conocido. Se considera día de quiebre todo día en que el stock disponible es nulo o negativo, contado a partir del lanzamiento del producto. De forma complementaria, a partir de los cambios diarios en el saldo de la bodega de producto vencido se estimaron las unidades que ingresan a ella, como medida directa de la pérdida por obsolescencia.

In [ ]:
# ----------------------------------------------------------------------------
# 1.7 (cont.) Disponibilidad de inventario comercial y obsolescencia
# ----------------------------------------------------------------------------
# BODEGAS_VENTA y BODEGA_VENCIDOS se definen en la celda de parámetros (validadas con el área de operaciones)
print("Saldos negativos en todas las bodegas:", int(pd_bodegas["saldos_negativos"].fillna(0).sum()))

# 1.7.c Stock diario en bodegas de venta por SKU (forward fill entre días con movimiento)
inv_com = (df_inventario.join(universo_ids, "ItemCode", "left_semi")
           .filter(F.col("Warehouse").isin(BODEGAS_VENTA))
           .select("ItemCode", "ItemCode_orig", "Warehouse", F.to_date("FechaCierre").alias("fecha"),
                   F.col("StockCierreDia").cast("double").alias("stock")))

fecha_min_inv = inv_com.agg(F.min("fecha")).first()[0]
fechas = spark.sql(f"SELECT explode(sequence(to_date('{fecha_min_inv}'), to_date('{FECHA_FIN}'), "
                   f"interval 1 day)) AS fecha")
w_ff = (Window.partitionBy("ItemCode_orig", "Warehouse").orderBy("fecha")
              .rowsBetween(Window.unboundedPreceding, 0))

stock_diario = (inv_com.select("ItemCode", "ItemCode_orig", "Warehouse").distinct().crossJoin(fechas)
                .join(inv_com, ["ItemCode", "ItemCode_orig", "Warehouse", "fecha"], "left")
                .withColumn("stock", F.last("stock", ignorenulls=True).over(w_ff))
                .fillna(0, subset=["stock"])
                .filter(F.col("fecha") >= F.to_date(F.lit(FECHA_INICIO)))
                .groupBy("ItemCode", "fecha").agg(F.sum("stock").alias("stock_comercial"))
                .cache())

sin_inv = universo_ids.join(stock_diario.select("ItemCode").distinct(), "ItemCode", "left_anti").count()
print(f"SKUs del universo sin registros en bodegas de venta: {sin_inv}")

# Periodo activo de cada SKU: desde su primer pedido o su primer stock positivo (lo que ocurra antes)
inicio_sku = spark.createDataFrame(pdu.loc[pdu["en_alcance"], ["ItemCode", "primer_pedido"]])
primer_stock = (stock_diario.filter("stock_comercial > 0")
                            .groupBy("ItemCode").agg(F.min("fecha").alias("primer_stock")))

disp = (stock_diario.join(inicio_sku, "ItemCode", "left").join(primer_stock, "ItemCode", "left")
        .withColumn("inicio_activo", F.least(F.to_date("primer_pedido"), F.col("primer_stock")))
        .filter(F.col("fecha") >= F.col("inicio_activo"))
        .withColumn("quiebre", (F.col("stock_comercial") <= 0).cast("int"))
        .withColumn("mes", F.trunc("fecha", "MM")))

quiebre_sku_mes = (disp.groupBy("ItemCode", "mes")
                       .agg(F.count("*").alias("dias_activos"), F.sum("quiebre").alias("dias_quiebre"))
                       .cache())   # se reutiliza en la sección 3.2

# Indicadores por SKU (últimos 12 meses)
pd_q12 = (quiebre_sku_mes.filter(F.col("mes") >= F.to_date(F.lit(INICIO_12M)))
          .groupBy("ItemCode")
          .agg(F.sum("dias_activos").alias("dias_activos_12m"), F.sum("dias_quiebre").alias("dias_quiebre_12m"))
          .toPandas())
pd_q12["pct_dias_quiebre_12m"] = pd_q12["dias_quiebre_12m"] / pd_q12["dias_activos_12m"] * 100
pd_perfil_sku = (pd_perfil_sku.drop(columns=["dias_activos_12m", "dias_quiebre_12m", "pct_dias_quiebre_12m"],
                                    errors="ignore")
                              .merge(pd_q12, on="ItemCode", how="left"))

display(pd_perfil_sku.groupby("abc_valor")
        .agg(skus=("ItemCode", "count"),
             pct_quiebre_mediana=("pct_dias_quiebre_12m", "median"),
             pct_quiebre_promedio=("pct_dias_quiebre_12m", "mean"),
             skus_sin_quiebre=("pct_dias_quiebre_12m", lambda s: (s == 0).sum()),
             skus_quiebre_mayor_10pct=("pct_dias_quiebre_12m", lambda s: (s > 10).sum()))
        .reset_index().round(2))

display(pd_perfil_sku[["pct_dias_quiebre_12m", "cv2_sem", "cv_12m", "tasa_devolucion_12m"]]
        .corr(method="spearman").loc[["pct_dias_quiebre_12m"]].round(3))

# Evolución mensual: % de días-SKU en quiebre por clase ABC
abc_sp = spark.createDataFrame(pd_perfil_sku[["ItemCode", "abc_valor"]])
pd_q_mes = (quiebre_sku_mes.join(abc_sp, "ItemCode")
            .groupBy("mes", "abc_valor")
            .agg(F.sum("dias_quiebre").alias("q"), F.sum("dias_activos").alias("a"))
            .toPandas())
pd_q_mes["pct"] = pd_q_mes["q"] / pd_q_mes["a"] * 100
pv_q = pd_q_mes.pivot(index="mes", columns="abc_valor", values="pct").sort_index()
pv_q.index = pd.to_datetime(pv_q.index)

fig, ax = plt.subplots(figsize=(11, 4))
for clase, color in colores.items():
    ax.plot(pv_q.index, pv_q[clase], marker="o", ms=3, color=color, label=f"Clase {clase}")
ax.set_ylabel("% de días-SKU en quiebre"); ax.set_title("Quiebres de stock en bodegas de venta por clase ABC")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.get_xticklabels(), rotation=45); ax.legend()
plt.tight_layout(); plt.show()

# 1.7.d Obsolescencia: ingresos netos diarios a la bodega de vencidos
w_v = Window.partitionBy("ItemCode_orig").orderBy("fecha")
ciclo_sp = spark.createDataFrame(pdu[["ItemCode", "ciclo_vida"]])
venc = (df_inventario.filter(F.col("Warehouse") == BODEGA_VENCIDOS)
        .select("ItemCode", "ItemCode_orig", F.to_date("FechaCierre").alias("fecha"),
                F.col("StockCierreDia").cast("double").alias("stock"))
        .withColumn("delta", F.col("stock") - F.lag("stock", 1, 0.0).over(w_v))
        .filter(F.col("fecha") >= F.to_date(F.lit(FECHA_INICIO)))
        .withColumn("ingreso", F.greatest("delta", F.lit(0.0)))
        .withColumn("mes", F.trunc("fecha", "MM"))
        .join(ciclo_sp, "ItemCode", "left")
        .fillna("6. Otro (no PT / sin pedidos)", subset=["ciclo_vida"])
        .cache())

# Por etapa de ciclo de vida (ventana completa y 12 meses)
pd_venc_ciclo = (venc.groupBy("ciclo_vida")
                 .agg(F.countDistinct(F.when(F.col("ingreso") > 0, F.col("ItemCode"))).alias("skus_con_vencimiento"),
                      F.sum("ingreso").alias("unid_vencidas_ventana"),
                      F.sum(F.when(F.col("mes") >= F.to_date(F.lit(INICIO_12M)), F.col("ingreso"))).alias("unid_vencidas_12m"))
                 .orderBy("ciclo_vida").toPandas())
pd_venc_ciclo["pct_unid_ventana"] = pd_venc_ciclo["unid_vencidas_ventana"] / pd_venc_ciclo["unid_vencidas_ventana"].sum() * 100
display(pd_venc_ciclo.round(1))

# Evolución mensual
pd_venc_mes = venc.groupBy("mes").agg(F.sum("ingreso").alias("unid_vencidas")).orderBy("mes").toPandas()
pd_venc_mes["mes"] = pd.to_datetime(pd_venc_mes["mes"])
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(pd_venc_mes["mes"], pd_venc_mes["unid_vencidas"], width=20, color="firebrick", alpha=0.8)
ax.set_ylabel("Unidades"); ax.set_title("Unidades que ingresan mensualmente a la bodega de vencidos")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.get_xticklabels(), rotation=45); plt.tight_layout(); plt.show()

# Por SKU del universo (12 meses), con valor estimado a precio promedio de venta
pd_v12 = (venc.filter(F.col("mes") >= F.to_date(F.lit(INICIO_12M)))
              .groupBy("ItemCode").agg(F.sum("ingreso").alias("unid_vencidas_12m")).toPandas())
pd_perfil_sku = (pd_perfil_sku.drop(columns=["unid_vencidas_12m", "tasa_vencimiento_12m"], errors="ignore")
                              .merge(pd_v12, on="ItemCode", how="left")
                              .fillna({"unid_vencidas_12m": 0}))
pd_perfil_sku["tasa_vencimiento_12m"] = np.where(
    pd_perfil_sku["unid_facturadas_12m"] > 0,
    pd_perfil_sku["unid_vencidas_12m"] / pd_perfil_sku["unid_facturadas_12m"] * 100, np.nan)
pd_perfil_sku["valor_vencido_est_12m"] = (pd_perfil_sku["unid_vencidas_12m"]
                                          * pd_perfil_sku["valor"] / pd_perfil_sku["unidades"])

display(pd_perfil_sku.groupby("abc_valor")
        .agg(skus_con_vencimiento=("unid_vencidas_12m", lambda s: (s > 0).sum()),
             unid_vencidas_12m=("unid_vencidas_12m", "sum"),
             valor_vencido_est_12m=("valor_vencido_est_12m", "sum"),
             tasa_venc_mediana=("tasa_vencimiento_12m", "median"))
        .reset_index().round(2))

display(pd_perfil_sku.sort_values("unid_vencidas_12m", ascending=False)
        [["ItemCode", "ItemName", "abc_valor", "sbc_mes", "unid_vencidas_12m", "tasa_vencimiento_12m",
          "valor_vencido_est_12m", "pct_dias_quiebre_12m", "tasa_devolucion_12m"]].head(15).round(2))

**Quiebres de stock.** El porcentaje de días en quiebre se resume por clase ABC y se sigue mes a mes. Lo esperable es una disponibilidad alta en los productos de mayor valor y un deterioro hacia los de menor valor. Los aumentos simultáneos en las tres clases suelen señalar eventos de abastecimiento que afectan a todo el portafolio y conviene documentarlos.

La correlación entre los días en quiebre y la variabilidad de la demanda tiene una lectura importante para el modelado. Durante el desabastecimiento la demanda registrada en los pedidos tiende a reducirse, porque los clientes no pueden pedir productos no disponibles o los compran a otro proveedor. La demanda observada en esos periodos está **censurada** y subestima la demanda real. Por eso los meses con quiebre se marcan explícitamente en la tabla base analítica, y su tratamiento se define en la fase de modelado. La medición se limita a las bodegas propias de venta: la disponibilidad en las bodegas de consignación no se considera.

**Obsolescencia.** Los ingresos a la bodega de vencidos se estiman a partir de los cambios diarios en su saldo y se separan por etapa de ciclo de vida, porque buena parte de lo que ingresa puede no ser producto terminado (insumos y materiales de la operación productiva) y queda fuera del alcance. Para el portafolio activo, las unidades vencidas se valoran a precio promedio de venta; ese valor es una cota superior del costo real de la pérdida.

Los productos que combinan vencimientos y quiebres en una misma ventana ilustran el problema que motiva este trabajo: no es solo de volumen, sino de oportunidad. Las compras y la rotación del inventario no están sincronizadas con la demanda, y eso es precisamente lo que un pronóstico confiable busca corregir.

### 1.8 Análisis por línea comercial.

La distribuidora organiza su portafolio en varias líneas comerciales, registradas en un campo personalizado del maestro de artículos. Esta sección caracteriza el universo de productos a ese nivel, con tres propósitos: identificar diferencias en importancia y comportamiento de la demanda entre líneas, evaluar si alguna línea responde a un modelo de negocio distinto que justifique un tratamiento diferenciado, y medir en qué grado la agregación por línea reduce la volatilidad observada a nivel de producto.

In [ ]:
# ----------------------------------------------------------------------------
# 1.8 Análisis por línea comercial
# ----------------------------------------------------------------------------
LINEA = "NombreLinea"
DIFICIL = ["Errática", "Intermitente", "Irregular"]
mapa_linea = pdu.loc[pdu["en_alcance"], ["ItemCode", LINEA]]
pl = pd_perfil_sku.merge(mapa_linea, on="ItemCode", how="left")

def skus_para_80(s):
    s = s.sort_values(ascending=False)
    return int((s.cumsum() / s.sum() < 0.8).sum() + 1)

# 1.8.a Perfil por línea
res_linea = pl.groupby(LINEA).agg(
    skus=("ItemCode", "count"),
    nuevos=("historia_corta", "sum"),
    valor_12m=("valor", "sum"),
    skus_A=("abc_valor", lambda s: (s == "A").sum()),
    skus_B=("abc_valor", lambda s: (s == "B").sum()),
    skus_C=("abc_valor", lambda s: (s == "C").sum()),
    skus_para_80pct_linea=("valor", skus_para_80),
    pct_patron_dificil_sem=("sbc_sem", lambda s: s.isin(DIFICIL).mean() * 100),
    cv12_mediano_sku=("cv_12m", "median"),
    share_top1_mediano=("share_top1_cliente", "median"),
    quiebre_mediano_12m=("pct_dias_quiebre_12m", "median"),
    devolucion_mediana_12m=("tasa_devolucion_12m", "median"),
)
res_linea["pct_valor"] = res_linea["valor_12m"] / res_linea["valor_12m"].sum() * 100

# 1.8.b Efecto de la agregación: CV de la serie de la línea vs. CV mediano de sus SKUs
serie_linea = (serie_mes.merge(mapa_linea, on="ItemCode")
                        .groupby([LINEA, "periodo"])["unidades"].sum().reset_index())
ult12_l = serie_linea[serie_linea["periodo"] >= pd.Timestamp(INICIO_12M)]
res_linea["cv12_serie_linea"] = ult12_l.groupby(LINEA)["unidades"].agg(lambda s: s.std(ddof=0) / s.mean())
res_linea["reduccion_cv_pct"] = (1 - res_linea["cv12_serie_linea"] / res_linea["cv12_mediano_sku"]) * 100

display(res_linea.sort_values("valor_12m", ascending=False).reset_index().round(2))

# Gráfica: evolución de cada línea (índice: 1 = promedio de la línea)
fig, ax = plt.subplots(figsize=(11, 4.5))
for linea, g in serie_linea.groupby(LINEA):
    ax.plot(g["periodo"], g["unidades"] / g["unidades"].mean(), marker="o", ms=2, label=linea)
ax.axhline(1, color="gray", ls=":", lw=1)
ax.set_ylabel("Índice (1 = promedio de la línea)")
ax.set_title("Evolución mensual de las unidades pedidas por línea comercial")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.get_xticklabels(), rotation=45); ax.legend(fontsize=8, ncol=3)
plt.tight_layout(); plt.show()

# 1.8.c Relación entre las líneas comerciales y la clasificación ABC (insumo de validación para la Fase 2)
display(pd.crosstab(pl[LINEA], pl["abc_valor"], margins=True).reset_index())

**Cómo leer los resultados.** El perfil por línea compara, para cada línea comercial, su número de productos, su participación en el valor, su composición ABC, la proporción de productos con patrón difícil y las medianas de variabilidad, dependencia de clientes, quiebres y devolución. Con él se responden tres preguntas:

1. *¿Hay diferencias de importancia y comportamiento entre líneas?* Es frecuente que una línea con pocos productos concentre una parte grande del valor con demanda muy estable, y que otras líneas tengan más renovación de portafolio, más variabilidad o más devoluciones.
2. *¿Alguna línea responde a un modelo de negocio distinto?* Una línea de producción por pedido (maquila) se reconoce en los datos porque la demanda de cada producto proviene de un único cliente, los productos permanecen casi siempre sin existencias en las bodegas de venta (su inventario se gestiona aparte) y la serie presenta picos aislados sin relación con la tendencia del negocio. Como su demanda depende de contratos y no del mercado, esa línea se excluye del alcance (`LINEAS_FUERA`, sección 1.9).
3. *¿Cuánto reduce la agregación la volatilidad?* El CV de la serie de cada línea se compara con el CV mediano de sus productos: las fluctuaciones individuales se compensan entre sí, y la reducción suele ser mayor en las líneas con más productos.

Aunque el pronóstico a nivel de línea sería más estable, la producción y las compras se planean por producto, de modo que la unidad de análisis se mantiene en el producto. La línea se conserva como variable de estratificación y como posible nivel superior de un esquema de pronóstico jerárquico en la Fase 3.

### 1.9 Delimitación del alcance.

Los hallazgos del análisis exploratorio permiten definir el universo de productos sobre el cual se desarrollarán la clasificación y el modelo de pronóstico. A partir del portafolio activo se aplican dos exclusiones adicionales. La primera es la línea de maquila, cuya producción se realiza por pedido para clientes específicos y cuya demanda depende de contratos y no del mercado (sección 1.8). La segunda son los productos con menos de doce meses de historia, cuya serie es insuficiente para caracterizar su variabilidad, estimar indicadores anuales y entrenar un modelo de pronóstico (sección 1.3). Estos productos nuevos requieren métodos específicos, como el pronóstico por analogía con productos similares, que quedan fuera del alcance de este trabajo. Las secciones siguientes (limpieza y tabla base analítica) se construyen exclusivamente sobre el universo resultante.

In [ ]:
# ----------------------------------------------------------------------------
# 1.9 Delimitación del alcance: universo de clasificación y pronóstico
# ----------------------------------------------------------------------------
# LINEAS_FUERA se define en la celda de parámetros (producción por pedido, sección 1.8)

pdu["excluido_linea"]   = pdu["NombreLinea"].isin(LINEAS_FUERA)
pdu["en_alcance_final"] = pdu["en_alcance"] & ~pdu["historia_corta"] & ~pdu["excluido_linea"]

def motivo_exclusion(r):
    if r["en_alcance_final"]:                     return "0. En alcance"
    if not r["en_maestro"]:                       return "1. Fuera del maestro PT"
    if r["es_muestra"]:                           return "2. Muestra gratuita"
    if r["ciclo_vida"].startswith(("3.", "4.")):  return "3. Sin demanda vigente"
    if r["excluido_linea"]:                       return "4. Línea por pedido"
    if r["historia_corta"]:                       return "5. Historia menor a 12 meses"
    return "6. Otro"

pdu["motivo_exclusion"] = pdu.apply(motivo_exclusion, axis=1)

embudo = (pdu.groupby("motivo_exclusion")
             .agg(skus=("ItemCode", "count"),
                  unidades=("unidades_pedidas", "sum"),
                  valor=("valor_pedido", "sum"))
             .sort_index())
for c in ["skus", "unidades", "valor"]:
    embudo[f"pct_{c}"] = embudo[c] / embudo[c].sum() * 100
display(embudo.reset_index().round(2))

# Redefinición del universo: todo lo que sigue (secciones 2 y 3) se construye sobre él
alcance       = pdu.loc[pdu["en_alcance_final"], "ItemCode"]
universo_ids  = spark.createDataFrame(alcance.to_frame())
pd_perfil_sku_activo = pd_perfil_sku.copy()   # perfil del portafolio activo (base de las secciones 1.4 a 1.8)
pd_perfil_sku = pd_perfil_sku[pd_perfil_sku["ItemCode"].isin(alcance)].reset_index(drop=True)

print(f"Universo final: {len(alcance)} SKUs | "
      f"{pdu.loc[pdu['en_alcance_final'], 'unidades_pedidas'].sum() / pdu['unidades_pedidas'].sum() * 100:.1f}% "
      f"de las unidades y "
      f"{pdu.loc[pdu['en_alcance_final'], 'valor_pedido'].sum() / pdu['valor_pedido'].sum() * 100:.1f}% "
      f"del valor pedido en la ventana")

**Cómo leer el embudo.** La tabla resume la delimitación del universo a partir del maestro de producto terminado: cuántos productos, unidades y valor se excluyen por cada motivo (fuera del maestro, muestras gratuitas, sin demanda vigente, línea por pedido e historia menor a doce meses). El criterio para aceptar el alcance es que el número de productos se reduzca de forma importante con una pérdida de demanda pequeña, y que todos los productos del universo final tengan historia suficiente para estimar los indicadores anuales de la tabla de características y entrenar el modelo de pronóstico.

# A1.2 — Calidad y limpieza de los datos.

A partir de los hallazgos del análisis exploratorio, se definió un conjunto de reglas de calidad que se aplican en orden secuencial sobre las tres fuentes documento. Para cada regla se registra el número de líneas, unidades y valor que excluye, de modo que el efecto acumulado de la limpieza sea verificable. Las reglas se dividen en dos tipos. Las reglas de exclusión eliminan registros que no representan demanda de producto (documentos de servicio, cancelaciones, líneas sin producto o con cantidades no válidas, y códigos fuera del alcance). Las reglas de marcación no eliminan registros, sino que identifican situaciones que requieren un tratamiento específico en las fases posteriores (líneas a precio cero y valores atípicos en las series de demanda).

In [ ]:
# ----------------------------------------------------------------------------
# 2. CALIDAD Y LIMPIEZA DE DATOS (A1.2)
# ----------------------------------------------------------------------------
QTY, VAL = F.col("Quantity").cast("double"), F.col("LineTotal").cast("double")
CALIDAD = []

def medir(fuente, paso, df):
    r = df.agg(F.count("*").alias("filas"), F.sum(QTY).alias("unidades"), F.sum(VAL).alias("valor")).first()
    CALIDAD.append({"fuente": fuente, "paso": paso, **r.asDict()})

def excluir(df, condicion):
    """Excluye las filas que cumplen la condición; un NULL en la condición NO excluye."""
    return df.filter(~F.coalesce(condicion, F.lit(False)))

# Códigos válidos: maestro PT sin muestras gratuitas
maestro_valido = spark.createDataFrame(pdu.loc[pdu["en_maestro"] & ~pdu["es_muestra"], ["ItemCode"]])

REGLAS = [
    ("R1 Documentos de servicio (DocType = 'S')",     lambda d: excluir(d, F.col("DocType") == "S")),
    ("R2 Cancelados y documentos de cancelación",     lambda d: excluir(d, F.col("CANCELED").isin("Y", "C"))),
    ("R3 Líneas sin ItemCode",                         lambda d: excluir(d, F.col("ItemCode").isNull())),
    ("R4 Líneas duplicadas (DocEntry, LineNum)",       lambda d: d.dropDuplicates(["DocEntry", "LineNum"])),
    ("R5 Cantidad nula o negativa",                    lambda d: excluir(d, F.col("Quantity") <= 0)),
    ("R6 Códigos fuera del maestro PT o muestras",     lambda d: d.join(maestro_valido, "ItemCode", "left_semi")),
]

limpio = {}
for fuente, df in [("Pedidos", df_pedidos), ("Ventas", df_ventas), ("Devoluciones", df_devoluciones)]:
    medir(fuente, "R0 Ventana de análisis (punto de partida)", df)
    for paso, regla in REGLAS:
        df = regla(df)
        medir(fuente, paso, df)
    limpio[fuente] = df.withColumn("precio_cero", (F.col("Price") == 0)).cache()

# 2.a Reporte de trazabilidad: filas, unidades y valor excluidos por regla
pd_cal = pd.DataFrame(CALIDAD)
for c in ["filas", "unidades", "valor"]:
    pd_cal[f"{c}_excluidas"] = pd_cal.groupby("fuente")[c].diff().mul(-1)
    pd_cal[f"pct_{c}_excluidas"] = pd_cal[f"{c}_excluidas"] / pd_cal.groupby("fuente")[c].transform("first") * 100
display(pd_cal[["fuente", "paso", "filas", "filas_excluidas", "pct_filas_excluidas",
                "unidades_excluidas", "pct_unidades_excluidas", "pct_valor_excluidas"]].round(3))

resumen_cal = pd_cal.groupby("fuente").agg(filas_ini=("filas", "first"), filas_fin=("filas", "last"),
                                           unid_ini=("unidades", "first"), unid_fin=("unidades", "last"))
resumen_cal["pct_filas_conservadas"] = resumen_cal["filas_fin"] / resumen_cal["filas_ini"] * 100
resumen_cal["pct_unid_conservadas"] = resumen_cal["unid_fin"] / resumen_cal["unid_ini"] * 100
display(resumen_cal.reset_index().round(2))

# 2.b Validación post-limpieza: nulos en campos clave y duplicados
for fuente, df in limpio.items():
    nulos = df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c)
                     for c in ["DocDate", "CardCode", "ItemCode", "Quantity", "Price"]]).first().asDict()
    dups = df.groupBy("DocEntry", "LineNum").count().filter("count > 1").count()
    print(f"{fuente:<13} nulos={nulos}  duplicados={dups}")

# 2.c Marcación: líneas a precio cero (posibles bonificaciones) en el universo
ped_lim_u = limpio["Pedidos"].join(universo_ids, "ItemCode", "left_semi")
pc = ped_lim_u.agg(F.avg(F.col("precio_cero").cast("int")).alias("pct_lineas"),
                   (F.sum(F.when(F.col("precio_cero"), QTY)) / F.sum(QTY)).alias("pct_unidades")).first()
print(f"\nLíneas a precio cero en el universo: {pc['pct_lineas']*100:.2f}% de las líneas, "
      f"{pc['pct_unidades']*100:.2f}% de las unidades")
display(ped_lim_u.groupBy("ItemCode")
        .agg(F.sum(QTY).alias("unidades"), F.sum(F.when(F.col("precio_cero"), QTY)).alias("unid_precio_cero"))
        .withColumn("pct_precio_cero", F.col("unid_precio_cero") / F.col("unidades") * 100)
        .filter("unid_precio_cero > 0").orderBy(F.desc("unid_precio_cero")).limit(15))

# 2.d Marcación: atípicos en la serie mensual de cada SKU (z robusto sobre residuo de mediana móvil)
Z_CORTE = 3.5   # Iglewicz & Hoaglin (1993)
pd_m_lim = (ped_lim_u.groupBy("ItemCode", F.trunc("DocDate", "MM").alias("periodo"))
                     .agg(F.sum(QTY).alias("unidades")).toPandas())
pd_m_lim["periodo"] = pd.to_datetime(pd_m_lim["periodo"])
serie_mes = densificar(pd_m_lim, meses)            # reemplaza la serie de la 1.5 por la versión limpia

def marcar_atipicos(g):
    g = g.sort_values("periodo").copy()
    g["base"] = g["unidades"].rolling(5, center=True, min_periods=3).median()
    r = g["unidades"] - g["base"]
    mad = (r - r.median()).abs().median()
    g["z_robusto"] = 0.6745 * (r - r.median()) / mad if mad and mad > 0 else 0.0
    return g

serie_mes = serie_mes.groupby("ItemCode", group_keys=False).apply(marcar_atipicos)
serie_mes["atipico"] = np.select([serie_mes["z_robusto"] > Z_CORTE, serie_mes["z_robusto"] < -Z_CORTE],
                                 ["Alto", "Bajo"], default="")

# Cruce con meses de quiebre: ¿los atípicos bajos coinciden con desabastecimiento?
pd_qm = quiebre_sku_mes.toPandas().rename(columns={"mes": "periodo"})
pd_qm["periodo"] = pd.to_datetime(pd_qm["periodo"])
serie_mes = (serie_mes.drop(columns=["dias_quiebre", "dias_activos"], errors="ignore")
                      .merge(pd_qm, on=["ItemCode", "periodo"], how="left")
                      .fillna({"dias_quiebre": 0}))
serie_mes["mes_con_quiebre"] = serie_mes["dias_quiebre"] > 0

res_at = (serie_mes[serie_mes["atipico"] != ""]
          .groupby("atipico")
          .agg(meses_sku=("ItemCode", "count"), skus=("ItemCode", "nunique"),
               pct_con_quiebre=("mes_con_quiebre", "mean")))
res_at["pct_con_quiebre"] *= 100
res_at["pct_del_total_meses_sku"] = res_at["meses_sku"] / len(serie_mes) * 100
display(res_at.reset_index().round(2))
print(f"% de meses-SKU con quiebre en toda la serie: {serie_mes['mes_con_quiebre'].mean()*100:.1f}%")

display(serie_mes[serie_mes["atipico"] != ""]
        .merge(pd_perfil_sku[["ItemCode", "ItemName", "abc_valor"]], on="ItemCode")
        .sort_values("z_robusto", key=abs, ascending=False)
        [["ItemCode", "ItemName", "abc_valor", "periodo", "unidades", "base", "z_robusto",
          "atipico", "dias_quiebre"]].head(20).round(2))

**Cómo leer el reporte de calidad.** Las reglas se aplican en orden: (R1) documentos de servicio, (R2) documentos cancelados y sus documentos de cancelación, (R3) líneas sin código de producto, (R4) líneas duplicadas, (R5) cantidades nulas o negativas y (R6) códigos fuera del maestro de producto terminado o muestras gratuitas. Para cada regla se reportan las filas, unidades y valor que excluye, y el resumen final muestra qué proporción de cada fuente se conserva.

Algunas lecturas útiles:

- En Devoluciones, R1 puede excluir una parte relevante del valor sin excluir unidades: son notas crédito por descuentos o ajustes comerciales, no mercancía devuelta.
- Si R3 no excluye ninguna línea después de R1 y R2, todas las líneas sin código correspondían a documentos de servicio o a cancelaciones, y no a errores de registro.
- R6 debería tener un efecto pequeño, porque los códigos reemplazados ya se homologaron a su código vigente.
- La validación posterior confirma que no quedan nulos en los campos clave ni líneas duplicadas.

**Marcaciones.** Las líneas a precio cero (posibles bonificaciones) se conservan como demanda si su peso no es material. Los valores atípicos de cada serie mensual se marcan con un puntaje z robusto sobre la desviación respecto de la mediana móvil de cinco meses (Iglewicz & Hoaglin, 1993), con umbral `Z_CORTE`.

El cruce de los atípicos con los meses de quiebre aporta evidencia sobre la censura de la demanda. Si la proporción de meses con quiebre es mayor entre los atípicos bajos que en el conjunto de la serie, las caídas anómalas coinciden con desabastecimiento. Si los atípicos altos también muestran más quiebres que la base, es consistente con un efecto de rebote: la demanda no atendida se registra cuando el producto vuelve a estar disponible. Siguiendo el principio de no eliminar información en la preparación, los atípicos se conservan marcados en la tabla base analítica y su tratamiento se define en la fase de modelado.

# A1.3 — Integración y Tabla Base Analítica.

##  3.1 Tabla mensual por producto (ABT_SKU_MES).

La tabla base analítica mensual integra, para cada producto del universo y cada mes desde su primer pedido, la demanda (variable objetivo), el resultado comercial (facturación y devoluciones), la disponibilidad de inventario (stock y días de quiebre), la obsolescencia (unidades vencidas) y las marcas de calidad definidas en la sección anterior. Los meses sin pedidos se registran explícitamente con demanda cero, de modo que cada serie sea continua en el tiempo. Esta tabla constituye el insumo del modelo predictivo de la Fase 3.

In [ ]:
# ----------------------------------------------------------------------------
# 3. INTEGRACIÓN Y TABLA BASE ANALÍTICA (A1.3)
# 3.1 ABT_SKU_MES: una fila por SKU y mes, desde su primer mes con demanda
# ----------------------------------------------------------------------------
def con_mes(df, col="DocDate"):
    return df.join(universo_ids, "ItemCode", "left_semi").withColumn("mes", F.trunc(col, "MM"))

def a_pandas(sdf):
    p = sdf.toPandas()
    p["mes"] = pd.to_datetime(p["mes"])
    return p

ped_m = con_mes(limpio["Pedidos"]).withColumn("ClienteID", F.regexp_replace("CardCode", r"-[^-]+$", ""))
ven_m = con_mes(limpio["Ventas"])
dev_m = con_mes(limpio["Devoluciones"])

agregados = {
    "pedidos": ped_m.groupBy("ItemCode", "mes").agg(
        F.sum(QTY).alias("unidades_pedidas"), F.sum(VAL).alias("valor_pedido"),
        F.countDistinct("DocEntry").alias("num_pedidos"), F.countDistinct("ClienteID").alias("num_clientes")),
    "ventas": ven_m.groupBy("ItemCode", "mes").agg(
        F.sum(QTY).alias("unidades_facturadas"), F.sum(VAL).alias("valor_facturado")),
    "devoluciones": dev_m.groupBy("ItemCode", "mes").agg(F.sum(QTY).alias("unidades_devueltas")),
    "stock": (stock_diario.withColumn("mes", F.trunc("fecha", "MM")).groupBy("ItemCode", "mes")
              .agg(F.avg("stock_comercial").alias("stock_promedio"),
                   F.expr("max_by(stock_comercial, fecha)").alias("stock_cierre_mes"))),
    "vencidos": (venc.join(universo_ids, "ItemCode", "left_semi")
                 .groupBy("ItemCode", "mes").agg(F.sum("ingreso").alias("unidades_vencidas"))),
}
TIENE_ENTREGAS = "DelivrdQty" in df_pedidos.columns
if TIENE_ENTREGAS:
    agregados["entregas"] = (ped_m.filter(F.col("LineStatus") == "C").groupBy("ItemCode", "mes")
        .agg(F.sum(F.least(F.col("DelivrdQty").cast("double"), QTY)).alias("unid_entregadas_cerradas"),
             F.sum(QTY).alias("unid_pedidas_cerradas")))
print(f"Cantidad entregada disponible en la extracción: {TIENE_ENTREGAS}")

# Columna vertebral: serie densa y limpia de la sección 2 (con atípicos y quiebres)
abt = (serie_mes.rename(columns={"periodo": "mes", "unidades": "unidades_serie", "base": "mediana_movil_5m"})
       [["ItemCode", "mes", "unidades_serie", "mediana_movil_5m", "z_robusto", "atipico",
         "dias_activos", "dias_quiebre"]])
for nombre, sdf in agregados.items():
    abt = abt.merge(a_pandas(sdf), on=["ItemCode", "mes"], how="left")

flujos = ["unidades_pedidas", "valor_pedido", "num_pedidos", "num_clientes", "unidades_facturadas",
          "valor_facturado", "unidades_devueltas", "unidades_vencidas"]
abt[flujos] = abt[flujos].fillna(0)

# Variables derivadas
abt["demanda_neta"]       = abt["unidades_facturadas"] - abt["unidades_devueltas"]
abt["ratio_fact_ped"]     = np.where(abt["unidades_pedidas"] > 0, abt["unidades_facturadas"] / abt["unidades_pedidas"], np.nan)
abt["precio_promedio"]    = np.where(abt["unidades_pedidas"] > 0, abt["valor_pedido"] / abt["unidades_pedidas"], np.nan)
abt["pct_dias_quiebre"]   = abt["dias_quiebre"] / abt["dias_activos"]
abt["mes_con_quiebre"]    = abt["dias_quiebre"].fillna(0) > 0
if TIENE_ENTREGAS:
    abt["fill_rate"] = np.where(abt["unid_pedidas_cerradas"] > 0,
                                abt["unid_entregadas_cerradas"] / abt["unid_pedidas_cerradas"], np.nan)
abt = abt.sort_values(["ItemCode", "mes"]).reset_index(drop=True)
abt["anio"]          = abt["mes"].dt.year
abt["mes_calendario"] = abt["mes"].dt.month
abt["mes_de_vida"]   = abt.groupby("ItemCode").cumcount() + 1       # 1 = primer mes con demanda
abt["mes_lanzamiento"] = abt["mes_de_vida"] == 1                   # suele ser un mes parcial

pd_abt_mes = (abt.drop(columns=["unidades_serie"])
                 .merge(pdu[["ItemCode", "NombreLinea"]], on="ItemCode", how="left"))

# Controles de consistencia
print(f"Filas: {len(pd_abt_mes):,} | SKUs: {pd_abt_mes['ItemCode'].nunique()} | "
      f"Meses: {pd_abt_mes['mes'].min():%Y-%m} a {pd_abt_mes['mes'].max():%Y-%m}")
print("Serie de la sección 2 = pedidos agregados:", np.allclose(abt["unidades_serie"], abt["unidades_pedidas"]))
tot_ped = ped_m.agg(F.sum(QTY)).first()[0]
tot_ven = ven_m.agg(F.sum(QTY)).first()[0]
print(f"Unidades pedidas   ABT / fuente limpia: {pd_abt_mes['unidades_pedidas'].sum():,.0f} / {tot_ped:,.0f}")
print(f"Unidades facturadas ABT / fuente limpia: {pd_abt_mes['unidades_facturadas'].sum():,.0f} / {tot_ven:,.0f} "
      f"(la diferencia es facturación anterior al primer pedido del SKU)")
print("Filas duplicadas SKU-mes:", pd_abt_mes.duplicated(["ItemCode", "mes"]).sum())

nulos_abt = pd_abt_mes.isna().sum()
display(nulos_abt[nulos_abt > 0].rename("nulos").reset_index().rename(columns={"index": "columna"}))
display(pd_abt_mes.describe().T.round(2).reset_index())

**Controles de la tabla.** La tabla debe conservar la totalidad de las unidades de las fuentes depuradas (la diferencia en unidades facturadas corresponde a facturación anterior al primer pedido del producto) y no debe tener registros duplicados por producto y mes. Los meses sin pedidos se conservan con demanda cero para mantener la continuidad de las series. Los valores faltantes se explican por construcción: las variables de precio y de nivel de servicio no están definidas en los meses sin pedidos.

El nivel de servicio se mide como la proporción de unidades entregadas en las líneas de pedido cerradas (`fill_rate`). La relación mensual entre unidades facturadas y pedidas se conserva solo como variable descriptiva, porque está afectada por el desfase entre la fecha del pedido y la de su facturación.

## 3.2 Actividad promocional y cambios de precio.

La distribuidora no cuenta con un registro estructurado de sus campañas promocionales, por lo que su efecto se aproximó a partir de los descuentos registrados en las líneas de pedido. Dado que los clientes de mayor tamaño tienen condiciones comerciales permanentes, el descuento promedio de un mes depende de la composición de clientes que compraron en él. Para aislar el componente promocional, se calculó el descuento incremental de cada línea como la diferencia entre el descuento aplicado y el descuento mediano que ese mismo cliente recibe habitualmente en ese producto. Un mes se considera promocional para un producto cuando su descuento incremental promedio, ponderado por unidades, supera un umbral definido. Adicionalmente, se identificaron los meses con alzas en el precio de lista, dado que los clientes tienden a anticipar sus compras ante un aumento de precios anunciado.

In [ ]:
# ----------------------------------------------------------------------------
# 3.2 Actividad promocional (proxy: descuento incremental) y cambios de precio de lista
# ----------------------------------------------------------------------------
assert {"DiscPrcnt", "PriceBefDi"} <= set(df_pedidos.columns), "Faltan DiscPrcnt / PriceBefDi en la extracción"

UMBRAL_PROMO_PP = 3.0     # puntos porcentuales de descuento incremental para marcar mes promocional
UMBRAL_ALZA_PCT = 3.0     # % de aumento del precio de lista mediano para marcar alza de precio
MIN_LINEAS_BASE = 3       # líneas mínimas cliente-SKU para estimar su descuento habitual

# 3.2.a Descuento por línea (calculado y declarado en SAP, para validar consistencia)
lin = (ped_m.filter(F.col("PriceBefDi") > 0)
       .withColumn("desc_pct", F.greatest(F.lit(0.0), F.least(F.lit(100.0),
                   (1 - F.col("Price").cast("double") / F.col("PriceBefDi").cast("double")) * 100))))

val = lin.agg(F.avg("desc_pct").alias("desc_calculado"),
              F.avg(F.col("DiscPrcnt").cast("double")).alias("desc_sap"),
              F.corr("desc_pct", F.col("DiscPrcnt").cast("double")).alias("correlacion"),
              F.avg((F.col("desc_pct") > 0).cast("int")).alias("pct_lineas_con_descuento")).first()
print(f"Descuento medio calculado: {val['desc_calculado']:.2f}% | declarado en SAP: {val['desc_sap']:.2f}% | "
      f"correlación: {val['correlacion']:.3f} | líneas con descuento: {val['pct_lineas_con_descuento']*100:.1f}%")

# 3.2.b Descuento habitual cliente-SKU y descuento incremental
base_cs = (lin.groupBy("ClienteID", "ItemCode")
              .agg(F.expr("percentile_approx(desc_pct, 0.5)").alias("desc_base_cs"),
                   F.count("*").alias("n_lineas_cs")))
lin = (lin.join(base_cs, ["ClienteID", "ItemCode"])
          .withColumn("desc_incremental",
                      F.when(F.col("n_lineas_cs") >= MIN_LINEAS_BASE, F.col("desc_pct") - F.col("desc_base_cs"))
                       .otherwise(F.lit(0.0))))

promo_mes = a_pandas(
    lin.groupBy("ItemCode", "mes").agg(
        (F.sum(QTY * F.col("desc_pct")) / F.sum(QTY)).alias("descuento_prom"),
        (F.sum(QTY * F.col("desc_incremental")) / F.sum(QTY)).alias("descuento_incremental"),
        (F.sum(F.when(F.col("desc_pct") > 0, QTY).otherwise(0.0)) / F.sum(QTY)).alias("pct_unid_con_descuento"),
        F.expr("percentile_approx(CAST(PriceBefDi AS DOUBLE), 0.5)").alias("precio_lista_mediano")))

cols_promo = ["descuento_prom", "descuento_incremental", "pct_unid_con_descuento", "precio_lista_mediano",
              "var_precio_lista_pct", "mes_promocional", "mes_alza_precio", "mes_previo_alza"]
pd_abt_mes = (pd_abt_mes.drop(columns=cols_promo, errors="ignore")
                        .merge(promo_mes, on=["ItemCode", "mes"], how="left")
                        .sort_values(["ItemCode", "mes"]).reset_index(drop=True))

# 3.2.c Marcas: mes promocional, alza de precio de lista y mes previo a un alza
pd_abt_mes["precio_lista_mediano"] = pd_abt_mes.groupby("ItemCode")["precio_lista_mediano"].ffill()
pd_abt_mes["var_precio_lista_pct"] = pd_abt_mes.groupby("ItemCode")["precio_lista_mediano"].pct_change(fill_method=None) * 100
pd_abt_mes["mes_promocional"]  = pd_abt_mes["descuento_incremental"].fillna(0) >= UMBRAL_PROMO_PP
pd_abt_mes["mes_alza_precio"]  = pd_abt_mes["var_precio_lista_pct"].fillna(0) >= UMBRAL_ALZA_PCT
pd_abt_mes["mes_previo_alza"]  = (pd_abt_mes.groupby("ItemCode")["mes_alza_precio"].shift(-1)
                                            .fillna(False).astype(bool))

# Sensibilidad del umbral promocional
display(pd.DataFrame([{"umbral_pp": u,
                       "pct_meses_sku_promo": (pd_abt_mes["descuento_incremental"].fillna(0) >= u).mean() * 100,
                       "skus_con_algun_mes_promo": pd_abt_mes.loc[pd_abt_mes["descuento_incremental"].fillna(0) >= u,
                                                                 "ItemCode"].nunique()}
                      for u in [1, 2, 3, 5, 10]]).round(2))

# 3.2.d Efecto sobre la demanda: relación con la mediana móvil y con los atípicos altos
pd_abt_mes["lift"] = pd_abt_mes["unidades_pedidas"] / pd_abt_mes["mediana_movil_5m"].replace(0, np.nan)

def efecto(marca):
    return (pd_abt_mes.groupby(marca)
            .agg(meses_sku=("ItemCode", "count"),
                 lift_mediano=("lift", "median"),
                 pct_atipico_alto=("atipico", lambda s: (s == "Alto").mean() * 100),
                 pct_con_quiebre=("mes_con_quiebre", "mean"))
            .assign(pct_con_quiebre=lambda d: d["pct_con_quiebre"] * 100)
            .reset_index().rename(columns={marca: "marca"}).assign(variable=marca))

display(pd.concat([efecto(m) for m in ["mes_promocional", "mes_alza_precio", "mes_previo_alza"]],
                  ignore_index=True)
          [["variable", "marca", "meses_sku", "lift_mediano", "pct_atipico_alto", "pct_con_quiebre"]].round(3))

# Sensibilidad del efecto promocional al umbral
sens = []
for u in [2, 3, 5, 10]:
    m = pd_abt_mes["descuento_incremental"].fillna(0) >= u
    sens.append({"umbral_pp": u, "pct_meses_sku": m.mean() * 100,
                 "lift_mediano_promo": pd_abt_mes.loc[m, "lift"].median(),
                 "lift_mediano_no_promo": pd_abt_mes.loc[~m, "lift"].median(),
                 "pct_atipico_alto_promo": (pd_abt_mes.loc[m, "atipico"] == "Alto").mean() * 100,
                 "pct_atipico_alto_no_promo": (pd_abt_mes.loc[~m, "atipico"] == "Alto").mean() * 100})
display(pd.DataFrame(sens).round(3))

# 3.2.e Evolución: % de SKUs en promoción y alzas de precio por mes
evo = (pd_abt_mes.groupby("mes")
       .agg(pct_skus_promo=("mes_promocional", "mean"), skus_con_alza=("mes_alza_precio", "sum"),
            descuento_prom=("descuento_prom", "mean")))
evo["pct_skus_promo"] *= 100

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(evo.index, evo["pct_skus_promo"], marker="o", ms=3, color="darkorange", label="% SKUs en promoción")
ax1.set_ylabel("% SKUs en promoción")
ax2 = ax1.twinx()
ax2.bar(evo.index, evo["skus_con_alza"], width=20, alpha=0.3, color="steelblue", label="SKUs con alza de precio")
ax2.set_ylabel("SKUs con alza de precio de lista"); ax2.grid(False)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left")
ax1.set_title("Actividad promocional y alzas de precio de lista por mes")
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax1.get_xticklabels(), rotation=45); plt.tight_layout(); plt.show()

**Cómo leer los resultados.**

- *Validación del indicador.* El descuento calculado con el precio antes y después de descuento debe coincidir con el porcentaje registrado en el ERP (correlación cercana a 1); de lo contrario, la construcción del indicador no es confiable.
- *Actividad promocional.* Se compara la demanda relativa a la mediana móvil (`lift`) y la proporción de atípicos altos entre meses promocionales y no promocionales. Un efecto que no se ve en la mediana pero sí en una mayor proporción de atípicos altos indica que la promoción actúa en episodios puntuales y no de forma generalizada. La tabla de sensibilidad verifica que la conclusión no dependa del umbral elegido. También conviene revisar si las promociones se concentran en periodos con disponibilidad, lo que es consistente con su uso para rotar el inventario.
- *Alzas del precio de lista.* Si las alzas se concentran en los primeros meses del año, corresponden a la actualización anual de las listas de precios. Una demanda por encima de lo habitual en el mes previo al alza y por debajo en el mes del alza es evidencia de compra anticipada, y explica parte de la estacionalidad estimada en la sección 1.2: no todo el patrón obedece al consumo, sino a la política de precios.
- *Implicaciones y limitaciones.* El descuento incremental y las marcas de promoción y de alza se incorporan a la tabla mensual como variables explicativas candidatas. La marca de mes previo a un alza requiere conocer con anticipación el calendario de alzas; como estas se anuncian a los clientes, esa información está disponible en la práctica, pero su uso debe controlarse en el modelado para evitar fuga de información. El indicador captura solo los descuentos de línea: los descuentos sobre el total del pedido y las promociones no monetarias (obsequios, exhibición) no quedan registrados.

## 3.3 Tabla de características por producto (ABT_SKU).

La tabla de características consolida, para cada producto del universo, los indicadores calculados a lo largo de la fase, organizados en ocho dimensiones: identificación, ciclo de vida, importancia para el negocio, patrón de demanda, dependencia de clientes, servicio e inventario, devoluciones y actividad comercial. Salvo que se indique lo contrario, los indicadores se calculan sobre los últimos doce meses de la ventana, de modo que reflejen la situación vigente del portafolio. Esta tabla constituye el insumo de la clasificación de productos de la Fase 2.

In [ ]:
# ----------------------------------------------------------------------------
# 3.3 ABT_SKU: tabla de características por producto (insumo de la Fase 2)
# ----------------------------------------------------------------------------
# 3.3.a Indicadores de 12 meses a partir de la ABT mensual
ref12 = pd_abt_mes[pd_abt_mes["mes"] >= pd.Timestamp(INICIO_12M)].copy()
ref12["w_desc"]     = ref12["descuento_prom"].fillna(0) * ref12["unidades_pedidas"]
ref12["w_desc_inc"] = ref12["descuento_incremental"].fillna(0) * ref12["unidades_pedidas"]
g12 = ref12.groupby("ItemCode")
s12 = g12[["unidades_pedidas", "valor_pedido", "unid_entregadas_cerradas",
           "unid_pedidas_cerradas", "w_desc", "w_desc_inc"]].sum()
den = s12["unidades_pedidas"].replace(0, np.nan)

feat = pd.DataFrame(index=s12.index)
feat["demanda_mensual_prom_12m"]  = g12["unidades_pedidas"].mean()
feat["precio_promedio_12m"]       = s12["valor_pedido"] / den
feat["fill_rate_12m"]             = s12["unid_entregadas_cerradas"] / s12["unid_pedidas_cerradas"].replace(0, np.nan)
feat["descuento_prom_12m"]        = s12["w_desc"] / den
feat["descuento_incremental_12m"] = s12["w_desc_inc"] / den
feat["pct_meses_promo_12m"]       = g12["mes_promocional"].mean() * 100
feat["pct_meses_atipicos_12m"]    = g12["atipico"].apply(lambda a: (a != "").mean() * 100)

# Crecimiento: últimos 6 meses vs. los 6 anteriores
ult6  = pd_abt_mes["mes"] >= (MES_FIN - 5).start_time
prev6 = (pd_abt_mes["mes"] >= (MES_FIN - 11).start_time) & ~ult6
u6 = pd_abt_mes[ult6].groupby("ItemCode")["unidades_pedidas"].sum()
p6 = pd_abt_mes[prev6].groupby("ItemCode")["unidades_pedidas"].sum()
feat["crecimiento_6m_pct"] = (u6 / p6.replace(0, np.nan) - 1) * 100

# Inventario al cierre de la ventana y cobertura en meses de demanda
feat["stock_final"] = (pd_abt_mes[pd_abt_mes["mes"] == MES_FIN.start_time]
                       .set_index("ItemCode")["stock_cierre_mes"])
feat["cobertura_meses"] = feat["stock_final"] / feat["demanda_mensual_prom_12m"].replace(0, np.nan)

# Historia completa
gh = pd_abt_mes.groupby("ItemCode")
feat["meses_historia"]         = gh.size()
feat["pct_meses_quiebre_hist"] = gh["mes_con_quiebre"].mean() * 100
feat = feat.reset_index()

# 3.3.b Ensamble y orden por dimensiones
abt_sku = (pd_perfil_sku.rename(columns={"valor": "valor_12m", "unidades": "unidades_12m"})
            .merge(pdu[["ItemCode", "ItmsGrpNam", "NombreLinea", "CreateDate", "ciclo_vida",
                       "primer_pedido", "ultimo_pedido"]],
                  on="ItemCode", how="left")
           .merge(feat, on="ItemCode", how="left"))

cols_cero = ["unid_facturadas_12m", "unid_devueltas_12m", "unid_vencidas_12m"]
abt_sku[cols_cero] = abt_sku[cols_cero].fillna(0)

DIMENSIONES = {
    "Identificación":        ["ItemCode", "ItemName", "ItmsGrpNam", "NombreLinea"],    
    "Ciclo de vida":         ["CreateDate", "primer_pedido", "ultimo_pedido", "meses_historia"],
    "Importancia":           ["valor_12m", "unidades_12m", "demanda_mensual_prom_12m",
                              "precio_promedio_12m", "crecimiento_6m_pct"],
    "Patrón de demanda":     ["pct_periodos_con_demanda_mes", "adi_mes", "cv2_mes",
                              "pct_periodos_con_demanda_sem", "adi_sem", "cv2_sem",
                              "cv_12m", "pct_meses_atipicos_12m"],
    "Clientes":              ["clientes_12m", "share_top1_cliente", "hhi_clientes"],
    "Servicio e inventario": ["fill_rate_12m", "pct_dias_quiebre_12m", "pct_meses_quiebre_hist", "stock_final",
                              "cobertura_meses", "unid_vencidas_12m", "tasa_vencimiento_12m",
                              "valor_vencido_est_12m"],
    "Devoluciones":          ["unid_facturadas_12m", "unid_devueltas_12m", "tasa_devolucion_12m"],
    "Comercial":             ["descuento_prom_12m", "descuento_incremental_12m", "pct_meses_promo_12m"],
}
columnas = [c for cols in DIMENSIONES.values() for c in cols]
faltan = [c for c in columnas if c not in abt_sku.columns]
print("Columnas esperadas que no existen:", faltan or "ninguna")
pd_abt_sku = abt_sku[[c for c in columnas if c in abt_sku.columns]].copy()

# 3.3.c Controles
print(f"SKUs: {len(pd_abt_sku)} | ItemCode únicos: {pd_abt_sku['ItemCode'].is_unique} | "
      f"Columnas: {pd_abt_sku.shape[1]}")
nulos_sku = pd_abt_sku.isna().sum()
display(nulos_sku[nulos_sku > 0].rename("nulos").reset_index().rename(columns={"index": "columna"}))
display(pd_abt_sku.describe().T.round(3).reset_index())

# 3.3.d Redundancia entre criterios candidatos para la clasificación (Spearman)
CRITERIOS = ["valor_12m", "demanda_mensual_prom_12m", "crecimiento_6m_pct", "cv_12m", "cv2_sem", "adi_sem",
             "clientes_12m", "share_top1_cliente", "fill_rate_12m", "pct_dias_quiebre_12m", "cobertura_meses",
             "tasa_devolucion_12m", "tasa_vencimiento_12m", "descuento_incremental_12m", "pct_meses_promo_12m"]
corr_crit = pd_abt_sku[CRITERIOS].corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_crit, annot=False, cmap="RdBu_r", vmin=-1, vmax=1, square=True,
            cbar_kws={"shrink": 0.7}, ax=ax)
for i in range(corr_crit.shape[0]):
    for j in range(corr_crit.shape[1]):
        v = corr_crit.iloc[i, j]
        ax.text(j + 0.5, i + 0.5, f"{v:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(v) > 0.6 else "black")
ax.set_title("Correlación de Spearman entre criterios candidatos de clasificación")
plt.tight_layout(); plt.show()

**Contenido de la tabla.** La tabla de características tiene una fila por producto del universo y variables organizadas en ocho dimensiones. De forma deliberada contiene solo variables continuas o descriptivas y no incluye etiquetas de clasificación (ABC, XYZ o patrón de demanda), porque la construcción de la clasificación corresponde a la Fase 2. Los faltantes se explican por la construcción de los indicadores (por ejemplo, el crecimiento semestral no se define sin demanda en el semestre de referencia, y las tasas de devolución y vencimiento no se definen sin facturación) y se conservan como faltantes; su tratamiento se decide en la Fase 2 según el método de clasificación.

La tabla captura a la vez los dos riesgos que motivan este trabajo: el desabastecimiento (días en quiebre) y el sobreinventario (cobertura en meses de demanda, que en productos de vida útil limitada es un riesgo directo de vencimiento).

**Redundancia entre criterios.** La matriz de Spearman entre los criterios candidatos permite identificar bloques que miden la misma dimensión subyacente, típicamente:

- *Tamaño del producto:* valor pedido, demanda mensual y número de clientes.
- *Irregularidad de la demanda:* variabilidad mensual, intermitencia semanal y porcentaje de días en quiebre, asociados negativamente con el tamaño (los productos de menor volumen son a la vez más irregulares y más propensos al desabastecimiento).
- *Actividad comercial:* descuento incremental y frecuencia de meses promocionales.

Los criterios con correlaciones débiles con los demás (crecimiento, cobertura, devolución, vencimiento, nivel de servicio y dependencia del cliente principal) aportan información propia. En la Fase 2 se retiene un criterio representativo por cada bloque redundante, junto con los criterios independientes.

Una relación débil entre el nivel de servicio y los días en quiebre es consistente con la censura de la demanda: durante el desabastecimiento se registran menos pedidos, de modo que el nivel de servicio, calculado sobre los pedidos registrados, se mantiene alto aunque el producto haya estado agotado. Por eso los días en quiebre son el indicador más confiable de la disponibilidad real.

In [ ]:
# ----------------------------------------------------------------------------
# 3.4 Persistencia en el Lakehouse (Delta) y diccionario de datos
# ----------------------------------------------------------------------------
import json
from datetime import datetime

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
META_PROCESO = {"snapshot": json.dumps(VERSIONES), "ventana": f"{FECHA_INICIO}/{FECHA_FIN}",
                "fecha_proceso": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
REGISTRO_TABLAS = []

def _pandas_a_spark(pdf):
    pdf = pdf.reset_index(drop=True).copy()
    for c in pdf.columns:
        if pdf[c].dtype == "object":
            no_nulos = pdf[c].dropna()
            if len(no_nulos) and no_nulos.map(lambda v: isinstance(v, (bool, np.bool_))).all():
                pdf[c] = pdf[c].astype("boolean")
            else:
                pdf[c] = pdf[c].astype("string")
    return spark.createDataFrame(pdf)

def guardar(df, nombre, descripcion=""):
    sdf = _pandas_a_spark(df) if isinstance(df, pd.DataFrame) else df
    sdf = (sdf.withColumn("_snapshot", F.lit(META_PROCESO["snapshot"]))
              .withColumn("_ventana", F.lit(META_PROCESO["ventana"]))
              .withColumn("_fecha_proceso", F.to_timestamp(F.lit(META_PROCESO["fecha_proceso"]))))
    sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(nombre)
    n = spark.table(nombre).count()
    REGISTRO_TABLAS.append({"tabla": nombre, "filas": n, "columnas": len(sdf.columns) - 3, "descripcion": descripcion})
    print(f"Guardada {nombre}: {n:,} filas")

# 3.4.a Diccionario de datos de las tablas Gold
DESC = {
    # --- Claves y calendario
    "ItemCode": "Código del producto (SKU) en SAP Business One",
    "mes": "Primer día del mes calendario de la observación",
    "anio": "Año calendario", "mes_calendario": "Mes calendario (1-12)",
    "mes_de_vida": "Meses transcurridos desde el primer mes con demanda del SKU (1 = primer mes)",
    "mes_lanzamiento": "Indica el primer mes con demanda del SKU (usualmente parcial)",
    "NombreLinea": "Línea comercial del producto (campo personalizado del maestro en SAP)",
    # --- Demanda y resultado comercial (ABT_SKU_MES)
    "unidades_pedidas": "Unidades pedidas por los clientes en el mes (variable objetivo)",
    "valor_pedido": "Valor de las líneas de pedido en moneda local (después de descuento de línea)",
    "num_pedidos": "Número de pedidos distintos que incluyen el SKU",
    "num_clientes": "Número de clientes distintos (NIT consolidado) que pidieron el SKU",
    "unidades_facturadas": "Unidades facturadas en el mes", "valor_facturado": "Valor facturado en moneda local",
    "unidades_devueltas": "Unidades devueltas físicamente en el mes",
    "demanda_neta": "Unidades facturadas menos unidades devueltas",
    "ratio_fact_ped": "Unidades facturadas / pedidas del mes (descriptiva; afectada por desfase pedido-factura)",
    "precio_promedio": "Valor pedido / unidades pedidas (moneda local por unidad)",
    "unid_pedidas_cerradas": "Unidades de las líneas de pedido cerradas del mes",
    "unid_entregadas_cerradas": "Unidades entregadas (DelivrdQty, acotadas a lo pedido) en líneas cerradas",
    "fill_rate": "Nivel de servicio: unidades entregadas / pedidas en líneas cerradas",
    # --- Inventario y obsolescencia
    "stock_promedio": "Stock promedio diario en bodegas de venta",
    "stock_cierre_mes": "Stock en bodegas de venta al último día del mes",
    "dias_activos": "Días del mes desde el inicio activo del SKU (primer pedido o primer stock)",
    "dias_quiebre": "Días del mes con stock nulo o negativo en bodegas de venta",
    "pct_dias_quiebre": "Días de quiebre / días activos",
    "mes_con_quiebre": "Indica si el mes tuvo al menos un día de quiebre (posible demanda censurada)",
    "unidades_vencidas": "Unidades que ingresan a la bodega de vencidos en el mes",
    # --- Calidad de la serie
    "mediana_movil_5m": "Mediana móvil centrada de 5 meses de las unidades pedidas",
    "z_robusto": "Puntaje z robusto del residuo frente a la mediana móvil",
    "atipico": "Marca de atípico: 'Alto', 'Bajo' o vacío (|z robusto| > 3,5)",
    "lift": "Unidades pedidas / mediana móvil de 5 meses",
    # --- Comercial (ABT_SKU_MES)
    "descuento_prom": "Descuento de línea promedio ponderado por unidades (%)",
    "descuento_incremental": "Descuento sobre el habitual de cada cliente para el SKU, ponderado por unidades (pp)",
    "pct_unid_con_descuento": "Fracción de las unidades pedidas con descuento de línea",
    "precio_lista_mediano": "Precio antes de descuento mediano del mes (arrastrado en meses sin pedidos)",
    "var_precio_lista_pct": "Variación % del precio de lista frente al mes anterior",
    "mes_promocional": "Descuento incremental >= 3 pp",
    "mes_alza_precio": "Aumento del precio de lista >= 3% frente al mes anterior",
    "mes_previo_alza": "El mes siguiente registra alza de precio (requiere calendario de alzas conocido)",
    # --- ABT_SKU: identificación y ciclo de vida
    "ItemName": "Descripción del producto", "ItmsGrpNam": "Grupo de artículos en SAP",
    "ciclo_vida": "Etapa de ciclo de vida en la ventana (activo con historia o nuevo)",
    "historia_corta": "Primer pedido dentro de los últimos 12 meses de la ventana",
    "CreateDate": "Fecha de creación del artículo en SAP",
    "primer_pedido": "Fecha del primer pedido en la ventana", "ultimo_pedido": "Fecha del último pedido en la ventana",
    "meses_historia": "Meses con registro en la ABT mensual desde el primer mes con demanda",
    # --- Importancia
    "valor_12m": "Valor pedido en los últimos 12 meses (moneda local)", "unidades_12m": "Unidades pedidas en los últimos 12 meses",
    "abc_valor": "Clase ABC por valor pedido, últimos 12 meses (80/95)",
    "abc_unidades": "Clase ABC por unidades pedidas, últimos 12 meses (80/95)",
    "demanda_mensual_prom_12m": "Unidades pedidas promedio por mes, últimos 12 meses",
    "precio_promedio_12m": "Precio promedio por unidad pedida, últimos 12 meses (moneda local)",
    "crecimiento_6m_pct": "Variación % de unidades pedidas: últimos 6 meses vs. 6 anteriores",
    # --- Patrón de demanda
    "pct_periodos_con_demanda_mes": "% de meses con demanda desde el primer mes del SKU",
    "adi_mes": "Intervalo promedio entre demandas, mensual (Syntetos-Boylan)",
    "cv2_mes": "CV² del tamaño de la demanda no nula, mensual",
    "sbc_mes": "Patrón SBC mensual: suave, errática, intermitente, irregular",
    "pct_periodos_con_demanda_sem": "% de semanas con demanda desde la primera semana del SKU",
    "adi_sem": "Intervalo promedio entre demandas, semanal", "cv2_sem": "CV² del tamaño de la demanda no nula, semanal",
    "sbc_sem": "Patrón SBC semanal: suave, errática, intermitente, irregular",
    "cv_12m": "Coeficiente de variación de la serie mensual, últimos 12 meses (incluye ceros)",
    "xyz": "Clase XYZ por cv_12m (X <= 0,5; Y <= 1,0; Z > 1,0)",
    "pct_meses_atipicos_12m": "% de meses marcados como atípicos, últimos 12 meses",
    # --- Clientes
    "clientes_12m": "Clientes distintos (NIT) con pedidos del SKU, últimos 12 meses",
    "share_top1_cliente": "Participación del cliente principal en las unidades del SKU, últimos 12 meses",
    "hhi_clientes": "Índice Herfindahl-Hirschman de concentración de clientes del SKU, últimos 12 meses",
    # --- Servicio e inventario
    "fill_rate_12m": "Unidades entregadas / pedidas en líneas cerradas, últimos 12 meses",
    "pct_dias_quiebre_12m": "% de días activos sin stock en bodegas de venta, últimos 12 meses",
    "dias_activos_12m": "Días activos del SKU en los últimos 12 meses",
    "dias_quiebre_12m": "Días de quiebre en los últimos 12 meses",
    "pct_meses_quiebre_hist": "% de meses con al menos un día de quiebre, historia completa",
    "stock_final": "Stock en bodegas de venta al cierre de la ventana",
    "cobertura_meses": "Stock final / demanda mensual promedio de 12 meses (meses de cobertura)",
    "unid_vencidas_12m": "Unidades ingresadas a la bodega de vencidos, últimos 12 meses",
    "tasa_vencimiento_12m": "Unidades vencidas / facturadas, últimos 12 meses (%)",
    "valor_vencido_est_12m": "Unidades vencidas valoradas a precio promedio de venta (cota superior, moneda local)",
    # --- Devoluciones y comercial (ABT_SKU)
    "unid_facturadas_12m": "Unidades facturadas, últimos 12 meses", "unid_devueltas_12m": "Unidades devueltas, últimos 12 meses",
    "tasa_devolucion_12m": "Unidades devueltas / facturadas, últimos 12 meses (%)",
    "descuento_prom_12m": "Descuento de línea promedio ponderado, últimos 12 meses (%)",
    "descuento_incremental_12m": "Descuento incremental promedio ponderado, últimos 12 meses (pp)",
    "pct_meses_promo_12m": "% de meses promocionales, últimos 12 meses",
}

def diccionario(pdf, tabla):
    return pd.DataFrame([{"tabla": tabla, "columna": c, "tipo": str(pdf[c].dtype),
                          "nulos": int(pdf[c].isna().sum()), "descripcion": DESC.get(c, "PENDIENTE")}
                         for c in pdf.columns])

pd_diccionario = pd.concat([diccionario(pd_abt_mes, "gld_abt_sku_mes"),
                            diccionario(pd_abt_sku, "gld_abt_sku")], ignore_index=True)
pendientes = pd_diccionario[pd_diccionario["descripcion"] == "PENDIENTE"]
print(f"Columnas sin descripción: {len(pendientes)}", pendientes["columna"].tolist() if len(pendientes) else "")

# 3.4.b Guardado: Silver (fuentes limpias), Aux (soporte y calidad), Gold (ABT)
guardar(pd_equivalencias, "aux_equivalencias_codigo",
          "Equivalencias de código viejo -> vigente, con validación por fechas de pedido")
guardar(limpio["Pedidos"].withColumn("ClienteID", F.regexp_replace("CardCode", r"-[^-]+$", "")),
        "slv_pedidos", "Líneas de pedido limpias (R1-R6), con ClienteID por NIT")
guardar(limpio["Ventas"], "slv_ventas", "Líneas de factura limpias (R1-R6)")
guardar(limpio["Devoluciones"], "slv_devoluciones", "Líneas de devolución limpias (R1-R6)")
guardar(stock_diario, "slv_stock_diario", "Stock diario por SKU del universo en bodegas de venta")
guardar(venc.select("ItemCode", "fecha", "mes", "stock", "delta", "ingreso", "ciclo_vida"),
        "slv_vencidos", "Saldo e ingresos netos diarios a la bodega de vencidos")
guardar(pdu[["ItemCode", "ItemName", "ItmsGrpNam", "validFor", "CreateDate", "OnHand", "en_maestro",
             "primer_pedido", "ultimo_pedido", "meses_con_pedido", "num_pedidos", "num_clientes",
             "unidades_pedidas", "valor_pedido", "ciclo_vida", "es_muestra", "en_alcance", "historia_corta",
             "NombreLinea", "excluido_linea", "en_alcance_final", "motivo_exclusion"]],
        "aux_universo_sku", "Maestro PT cruzado con demanda: ciclo de vida, muestras y alcance")
guardar(pd_cal, "aux_calidad", "Trazabilidad de la limpieza por fuente y regla (reporte de calidad)")
guardar(pd_diccionario, "aux_diccionario", "Diccionario de datos de las tablas Gold")
guardar(pd_abt_mes, "gld_abt_sku_mes", "ABT mensual por SKU: demanda, servicio, inventario, comercial y marcas")
guardar(pd_abt_sku, "gld_abt_sku", "ABT de características por SKU: insumo de la clasificación (Fase 2)")

display(pd.DataFrame(REGISTRO_TABLAS))
display(pd_diccionario)

## Conclusiones de la fase de comprensión y preparación de los datos.

La primera fase construye, a partir de los registros transaccionales del ERP, una base analítica depurada, trazable y documentada. Sobre un snapshot fijo de los datos y una ventana de meses completos, se integran cinco fuentes del ERP (pedidos, facturas, devoluciones, inventario y maestro de productos) y una tabla de equivalencias que homologa los cambios de código de producto. Se aplican seis reglas de calidad con trazabilidad por regla y se obtienen dos tablas analíticas: la tabla mensual por producto (`gld_abt_sku_mes`), insumo del modelo predictivo, y la tabla de características por producto (`gld_abt_sku`), insumo de la clasificación.

Los hallazgos metodológicos que orientan las fases siguientes son:

- **El universo relevante es mucho menor que el catálogo.** Se excluyen los productos sin demanda vigente, las muestras gratuitas, la línea de producción por pedido, cuya demanda es contractual, y los productos con menos de un año de historia, con una pérdida de demanda pequeña.
- **La homologación de códigos es indispensable.** Una recodificación de referencias fragmenta la historia de un producto en dos series; sin corregirla, el portafolio parece más extenso y las transiciones de código se interpretan como volatilidad o desabastecimiento.
- **La importancia de un producto no determina qué tan predecible es su demanda.** Hay productos de alto valor con patrones erráticos o irregulares, y la priorización por un único criterio cambia según el criterio y el periodo. Ambos resultados justifican una clasificación multicriterio que combine importancia y pronosticabilidad, fundamento del enfoque híbrido propuesto.
- **La volatilidad tiene causas identificables.** La dependencia de pocos clientes, los quiebres de stock, las promociones y las alzas anuales de precio explican parte de las fluctuaciones de la demanda y quedan registradas como variables en la tabla mensual.
- **La demanda observada está censurada por la disponibilidad.** Los meses con quiebre coinciden con caídas anómalas de la demanda con más frecuencia de la esperada, seguidas de rebotes cuando el producto vuelve a estar disponible. Estos meses quedan marcados para su tratamiento en el modelado.
- **La granularidad mensual es la adecuada.** A nivel mensual la mayor parte de la demanda sigue patrones regulares, mientras que a nivel semanal una proporción alta de productos requeriría métodos especializados para demanda errática o intermitente.

Con estos insumos, la Fase 2 selecciona los criterios no redundantes de la tabla de características para segmentar el universo, utiliza los métodos ABC, XYZ y de patrones de demanda como referencias de comparación y prioriza el grupo piloto de productos sobre el cual se desarrolla el modelo de predicción de demanda.